
# Evaluating Low-Cost PM2.5 Sensors
## Calibration Accuracy, Generalization, and Reliability

**Course:** MSDS 696 — Data Science Practicum II  
**Project track:** Curated Topic 15 — Air Quality  
**Stage:** Week 2 — Data Profiling and Quality Assessment  
**Data download date:** October 22, 2026

---

## 1. Project Overview

Low-cost air-quality sensors can expand the spatial coverage of particulate matter (PM2.5) monitoring. However, their measurements may differ from those of reference monitoring instruments due to sensor characteristics, environmental conditions, and other sources of measurement error.

This project evaluates agreement between low-cost PurpleAir sensor measurements and reference monitor measurements. It investigates whether calibration models improve this agreement and whether their performance holds under wildfire-smoke conditions.

### Research Question

How much does calibration improve agreement between low-cost PurpleAir sensors and reference monitors, and does the improvement hold under wildfire-smoke conditions?

### Project Objectives

1. Profile the available datasets and document potential data-quality issues.
2. Characterize agreement between raw PurpleAir measurements and reference monitor measurements.
3. Compare raw measurements with independently evaluated calibration models.
4. Evaluate generalization using a documented holdout strategy.
5. Assess performance under wildfire-smoke conditions.

---

## 2. Data Sources

The project uses two source datasets. They will initially be profiled separately because their temporal resolution, structure, and observation units may differ.

**Dataset A — 24-hour collocation dataset**

- Source: [EPA Pasteur data repository](https://pasteur.epa.gov/uploads/10.23719/1522388/Data_DevelopmentUSPAcorrection_210408.zip)
- Main file: `Full24hrdataset.csv`
- Intended role: Compare daily PurpleAir measurements with reference monitor measurements.

**Dataset B — Hourly wildfire-smoke dataset**

- Source: [EPA Pasteur data repository](https://pasteur.epa.gov/uploads/10.23719/1528138/Dataset_PurpleAirSmoke_Barkjohn_3_15_23.zip)
- Main file: `Finaldataset_correcteddatasetwholding_10_26_22.csv`
- Intended role: Evaluate measurement agreement and calibration performance under wildfire-smoke conditions.

The original files are stored locally under `data/raw/`. The raw files will remain unchanged.

---

## 3. Purpose of This Notebook

This notebook establishes a documented understanding of the source data before cleaning, feature engineering, or model training.

The profiling process will examine:

- File availability, dataset size, and structure
- Column names, definitions, and data types
- The meaning of one row and uniqueness of candidate identifiers
- Date coverage and unexpected dates
- Missingness in key variables
- Potential data-quality issues and proposed handling decisions
- Differences in missingness across relevant groups

The notebook records observations and proposed decisions separately. A suspicious value will not automatically be treated as an error.

---

## 4. Reproducibility and Data Integrity

- Use project-relative file paths rather than machine-specific paths.
- Preserve downloaded source files.
- Document the source, download date, and file names.
- Investigate duplicate records before deciding whether they are valid repeated observations.
- Keep the two datasets separate unless a justified integration strategy is established.
- Do not use reference measurements as predictors when those same measurements are the prediction target.
- Prevent information leakage when constructing training, validation, and test sets.
- Fit data-dependent preprocessing on training data only.

**Current status:** Initial data profiling. No cleaning, model training, or performance evaluation has been completed.



## 5. Environment and Project Setup

The following cells import the libraries required for initial data profiling and locate the project root dynamically.

The notebook assumes the repository has the following structure:

```text
low-cost-pm25-calibration/
├── data/
│   ├── raw/
│   │   ├── 24hr_collocation/
│   │   └── hourly_smoke/
│   └── processed/
├── notebooks/
│   └── 01_data_profile.ipynb
├── src/
│   └── download_data.py
├── reports/
├── .gitignore
├── requirements.txt
└── README.md
```

Raw source data is excluded from Git commits. A user reproducing the project can run `python src/download_data.py` from the repository root to prepare the datasets.


In [1]:

from pathlib import Path
import pandas as pd

# Find the project root from the notebook's current directory.
current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook "
        "from inside the low-cost-pm25-calibration repository."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"

COLLOCATION_DIR = RAW_DIR / "24hr_collocation"
SMOKE_DIR = RAW_DIR / "hourly_smoke"

COLLOCATION_FILE = COLLOCATION_DIR / "Full24hrdataset.csv"
SMOKE_FILE = (
    SMOKE_DIR
    / "Finaldataset_correcteddatasetwholding_10_26_22.csv"
)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)


Project root: .
Raw data directory: data/raw


In [2]:

# Verify the main CSV files before attempting to load them.
required_files = {
    "24-hour collocation dataset": COLLOCATION_FILE,
    "Hourly wildfire-smoke dataset": SMOKE_FILE,
}

for dataset_name, file_path in required_files.items():
    if not file_path.is_file():
        raise FileNotFoundError(
            f"{dataset_name} is missing: {file_path}\n"
            "Run `python src/download_data.py` from the project root."
        )

    print(
        f"{dataset_name}: available "
        f"({file_path.stat().st_size:,} bytes)"
    )


24-hour collocation dataset: available (6,420,524 bytes)
Hourly wildfire-smoke dataset: available (261,969,901 bytes)



## 6. Dataset A — 24-Hour Collocation Data

This dataset contains daily collocation observations used to compare low-cost PurpleAir sensor measurements with reference monitor measurements.

Before using it for analysis, we will verify its actual dimensions, inspect its variables, establish the observation unit, and assess potential data-quality issues.

The first inspection is descriptive only. No records will be removed or modified.


In [3]:
# Load the main 24-hour CSV without modifying the source file.
df_24hr = pd.read_csv(COLLOCATION_FILE, low_memory=False)

print(f"Number of rows: {len(df_24hr):,}")
print(f"Number of columns: {df_24hr.shape[1]:,}")


Number of rows: 12,430
Number of columns: 31


In [4]:

print("Columns in the 24-hour dataset:")

for column in df_24hr.columns:
    print(f"- {column}")


Columns in the 24-hour dataset:
- Unnamed: 0
- Date
- ID
- region
- FEMorFRM
- FEMorFRMtype
- PM25FM
- PM25cfatm
- PM25cf1
- b0.3um
- b0.5um
- b1um
- b2.5um
- b5um
- b10um
- b0.3to0.5
- b0.5to1
- b1to2.5
- b2.5to5
- b5to10
- TempC
- RH
- Dewpoint
- PM25PAlinearcor
- PM25PARHcor
- ratioPARHcor_FM
- diffPARHcor_FM
- ratiocfatm_FM
- diffcfatm_FM
- ratiocf1_FM
- diffcf1_FM


In [5]:

# View the first five records to understand the table structure.
display(df_24hr.head())


,Unnamed: 0,Date,ID,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cfatm,PM25cf1,b0.3um,...,RH,Dewpoint,PM25PAlinearcor,PM25PARHcor,ratioPARHcor_FM,diffPARHcor_FM,ratiocfatm_FM,diffcfatm_FM,ratiocf1_FM,diffcf1_FM
0,1,2019-05-17,AK1,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,6.7,8.257688,8.616954,1154.620023,...,38.564815,3.629662,6.033138,6.945963,1.036711,0.245963,1.232491,1.557688,1.286112,1.916954
1,2,2019-05-18,AK1,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,3.8,3.359005,3.493916,603.133763,...,49.404315,5.442318,3.351630,3.327497,0.875657,-0.472503,0.883949,-0.440995,0.919452,-0.306084
2,3,2019-05-21,AK1,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,4.0,3.799601,3.799601,626.582347,...,29.972222,1.734051,3.511632,5.161791,1.290448,1.161791,0.949900,-0.200399,0.949900,-0.200399
3,4,2019-05-22,AK1,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,4.7,4.369691,4.369691,723.777519,...,32.474513,4.165624,3.810029,5.244959,1.115949,0.544959,0.929721,-0.330309,0.929721,-0.330309
4,5,2019-05-19,AK1,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,3.2,3.191071,3.191071,505.051821,...,43.898226,5.867611,3.193114,3.643155,1.138486,0.443155,0.997210,-0.008929,0.997210,-0.008929


In [6]:

# Create an initial structural summary for every column.
profile_24hr = pd.DataFrame({
    "data_type": df_24hr.dtypes.astype(str),
    "non_null_count": df_24hr.notna().sum(),
    "missing_count": df_24hr.isna().sum(),
    "missing_percent": (
        df_24hr.isna().mean() * 100
    ).round(2),
})

display(profile_24hr)


,data_type,non_null_count,missing_count,missing_percent
Unnamed: 0,int64,12430,0,0.00
Date,str,12430,0,0.00
ID,str,12430,0,0.00
region,str,12430,0,0.00
FEMorFRM,str,12430,0,0.00
FEMorFRMtype,str,12430,0,0.00
PM25FM,float64,12430,0,0.00
PM25cfatm,float64,12430,0,0.00
PM25cf1,float64,12430,0,0.00
b0.3um,float64,12430,0,0.00



### 6.1 What Does One Row Represent?

A row should not be described as one unique sensor-day until the available identifiers, timestamps, and documentation support that interpretation.

We will identify candidate date and sensor/site columns, inspect repeated identifier combinations, and distinguish legitimate repeated observations from possible duplicate records.

A repeated sensor-date combination is a signal for investigation, not proof that a row should be deleted.


In [7]:

# Identify possible date, time, and sensor/site identifier fields.
# These are candidates based on column names, not confirmed definitions.
candidate_terms = (
    "date", "time", "day", "hour", "sensor", "site",
    "location", "device", "id", "region", "state"
)

candidate_columns_24hr = [
    column
    for column in df_24hr.columns
    if any(term in column.lower() for term in candidate_terms)
]

print("Candidate date/time/identifier columns:")
print(candidate_columns_24hr)

if candidate_columns_24hr:
    display(df_24hr[candidate_columns_24hr].head(10))


Candidate date/time/identifier columns:
['Date', 'ID', 'region']


,Date,ID,region
0,2019-05-17,AK1,Alaska
1,2019-05-18,AK1,Alaska
2,2019-05-21,AK1,Alaska
3,2019-05-22,AK1,Alaska
4,2019-05-19,AK1,Alaska
5,2019-05-20,AK1,Alaska
6,2019-05-25,AK1,Alaska
7,2019-05-26,AK1,Alaska
8,2019-05-23,AK1,Alaska
9,2019-05-24,AK1,Alaska


In [8]:

# Check how many regions are associated with each sensor ID.
regions_per_id = (
    df_24hr.groupby("ID", dropna=False)["region"]
    .nunique(dropna=False)
)

print("Number of distinct sensor IDs:", df_24hr["ID"].nunique())
print("IDs associated with multiple regions:",
      (regions_per_id > 1).sum())

print("\nNumber of rows per region:")
display(
    df_24hr["region"]
    .value_counts(dropna=False)
    .rename_axis("region")
    .reset_index(name="row_count")
)


Number of distinct sensor IDs: 49
IDs associated with multiple regions: 0

Number of rows per region:


,region,row_count
0,North,5762
1,West,3762
2,Central Southwest,1504
3,Southeast,1033
4,Alaska,369



### 6.2 Initial Uniqueness Checks

We will check whether complete rows are repeated exactly. We will also investigate whether plausible combinations of date and sensor/site identifiers repeat, once those columns have been identified.

Exact duplicate rows and repeated sensor-date combinations are different situations. Two records can share a sensor and date but contain different measurements or represent different sampling events.


In [9]:

# Count exact duplicate records without removing any rows.
exact_duplicate_count_24hr = int(df_24hr.duplicated().sum())

print(
    "Exact duplicate rows:",
    f"{exact_duplicate_count_24hr:,}"
)


Exact duplicate rows: 0


In [10]:

# Check whether the same sensor ID and date occur more than once.
# Include region to help identify possible inconsistencies in location labels.
sensor_date_keys = ["ID", "Date", "region"]

sensor_date_counts = (
    df_24hr.groupby(sensor_date_keys, dropna=False)
    .size()
    .reset_index(name="row_count")
)

repeated_sensor_dates = sensor_date_counts[
    sensor_date_counts["row_count"] > 1
].sort_values("row_count", ascending=False)

print("Distinct ID-date-region combinations:",
      len(sensor_date_counts))
print("Combinations appearing more than once:",
      len(repeated_sensor_dates))
print("Rows belonging to repeated combinations:",
      repeated_sensor_dates["row_count"].sum())
print("Extra rows beyond one per combination:",
      (repeated_sensor_dates["row_count"] - 1).sum())

display(repeated_sensor_dates.head(15))


Distinct ID-date-region combinations: 9707
Combinations appearing more than once: 2199
Rows belonging to repeated combinations: 4922
Extra rows beyond one per combination: 2723


,ID,Date,region,row_count
5191,IA11,2019-02-20,North,4
6227,IA3,2019-07-08,North,4
5974,IA3,2018-05-02,North,4
5030,IA1,2019-11-17,North,4
6336,IA4,2019-02-02,North,4
6147,IA3,2019-02-20,North,4
5610,IA2,2019-02-02,North,4
5564,IA2,2018-11-22,North,4
5071,IA10,2019-02-02,North,4
5575,IA2,2018-12-10,North,4


In [11]:

# Inspect the most frequently repeated sensor-date combinations.
# Show all columns to identify which values differ across records.
top_repeated_keys = repeated_sensor_dates.head(3)[
    ["ID", "Date", "region"]
]

for _, key in top_repeated_keys.iterrows():
    mask = (
        df_24hr["ID"].eq(key["ID"])
        & df_24hr["Date"].eq(key["Date"])
        & df_24hr["region"].eq(key["region"])
    )

    print(
        f"\nID={key['ID']}, Date={key['Date']}, "
        f"Region={key['region']}"
    )

    display(df_24hr.loc[mask].T)



ID=IA11, Date=2019-02-20, Region=North


,8692,8765,8986,9052
Unnamed: 0,8693,8766,8987,9053
Date,2019-02-20,2019-02-20,2019-02-20,2019-02-20
ID,IA11,IA11,IA11,IA11
region,North,North,North,North
FEMorFRM,FRM,FRM,FEM,FEM
FEMorFRMtype,R & P Model 2025 PM-2.5 Sequential Air Sampler...,R & P Model 2025 PM-2.5 Sequential Air Sampler...,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,Teledyne T640 at 5.0 LPM - Broadband spectroscopy
PM25FM,9.5,8.9,12.625,11.629167
PM25cfatm,27.407415,27.407415,27.407415,27.407415
PM25cf1,28.075102,28.075102,28.075102,28.075102
b0.3um,2253.086925,2253.086925,2253.086925,2253.086925



ID=IA3, Date=2019-07-08, Region=North


,7820,8058,8185,8296
Unnamed: 0,7821,8059,8186,8297
Date,2019-07-08,2019-07-08,2019-07-08,2019-07-08
ID,IA3,IA3,IA3,IA3
region,North,North,North,North
FEMorFRM,FRM,FEM,FRM,FEM
FEMorFRMtype,R & P Model 2025 PM-2.5 Sequential Air Sampler...,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,R & P Model 2025 PM-2.5 Sequential Air Sampler...,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...
PM25FM,17.5,17.0,17.2,17.5
PM25cfatm,42.13134,42.13134,42.13134,42.13134
PM25cf1,49.677972,49.677972,49.677972,49.677972
b0.3um,4832.846132,4832.846132,4832.846132,4832.846132



ID=IA3, Date=2018-05-02, Region=North


,7931,7951,8359,8361
Unnamed: 0,7932,7952,8360,8362
Date,2018-05-02,2018-05-02,2018-05-02,2018-05-02
ID,IA3,IA3,IA3,IA3
region,North,North,North,North
FEMorFRM,FEM,FRM,FEM,FRM
FEMorFRMtype,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,R & P Model 2025 PM-2.5 Sequential Air Sampler...,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,R & P Model 2025 PM-2.5 Sequential Air Sampler...
PM25FM,8.272727,9.6,9.347826,9.3
PM25cfatm,21.548538,21.548538,21.548538,21.548538
PM25cf1,23.264915,23.264915,23.264915,23.264915
b0.3um,2850.623353,2850.623353,2850.623353,2850.623353



### 6.3 Date Coverage and Variable Definitions

The date range and the definitions of key variables will be verified using the actual field names and source documentation.

The dataset's filename or publication date does not necessarily indicate the period covered by its observations. We will inspect the date field rather than infer the observation period from the filename.


In [12]:

# Display possible date/time columns and their example values.
date_columns_24hr = [
    column for column in df_24hr.columns
    if any(
        term in column.lower()
        for term in ("date", "time", "day", "year")
    )
]

print("Potential date/time columns:", date_columns_24hr)

for column in date_columns_24hr:
    print(f"\n{column}")
    print(df_24hr[column].head(10).to_list())


Potential date/time columns: ['Date']

Date
['2019-05-17', '2019-05-18', '2019-05-21', '2019-05-22', '2019-05-19', '2019-05-20', '2019-05-25', '2019-05-26', '2019-05-23', '2019-05-24']


In [13]:

# Parse dates in a separate variable; do not change the original dataset.
dates_24hr = pd.to_datetime(df_24hr["Date"], errors="coerce")

print("Date values that could not be parsed:", dates_24hr.isna().sum())
print("First observation date:", dates_24hr.min())
print("Last observation date:", dates_24hr.max())
print("Number of distinct dates:", dates_24hr.nunique())
print("Rows with dates outside 2017–2020:",
      ((dates_24hr.dt.year < 2017) |
       (dates_24hr.dt.year > 2020)).sum())


Date values that could not be parsed: 0
First observation date: 2017-09-30 00:00:00
Last observation date: 2020-01-13 00:00:00
Number of distinct dates: 830
Rows with dates outside 2017–2020: 0



### 6.4 Key Variables and Missingness

The next step is to identify the reference measurement, raw PurpleAir measurement, relative humidity, and any relevant sensor/site identifier.

Variable meanings will be verified against the dataset documentation. Missingness percentages will be recorded for the selected variables and used to investigate potential data-quality problems.


In [14]:

# Rank columns by missing-value percentage.
missingness_24hr = (
    df_24hr.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .to_frame()
)

display(missingness_24hr)


,missing_percent
Dewpoint,1.480290
TempC,1.480290
ratioPARHcor_FM,1.335479
ratiocf1_FM,1.198713
ratiocfatm_FM,1.198713
diffPARHcor_FM,0.136766
PM25PARHcor,0.136766
RH,0.136766
Unnamed: 0,0.000000
b0.5to1,0.000000


In [15]:

# Examine non-positive measurements without deleting or changing rows.
measurement_columns = ["PM25FM", "PM25cfatm", "PM25cf1"]

measurement_checks = []

for column in measurement_columns:
    values = df_24hr[column]

    measurement_checks.append({
        "column": column,
        "zero_count": int(values.eq(0).sum()),
        "negative_count": int(values.lt(0).sum()),
        "non_positive_count": int(values.le(0).sum()),
        "minimum": values.min(),
        "maximum": values.max(),
    })

display(pd.DataFrame(measurement_checks))


,column,zero_count,negative_count,non_positive_count,minimum,maximum
0,PM25FM,1,43,44,-1.777778,108.800000
1,PM25cfatm,0,0,0,0.181708,169.907631
2,PM25cf1,0,0,0,0.181708,256.309609


In [16]:

# Inspect records with non-positive reference readings.
# These are flagged for review, not automatically removed.
nonpositive_reference = df_24hr[
    df_24hr["PM25FM"].le(0)
]

print("Rows with PM25FM <= 0:", len(nonpositive_reference))

display(
    nonpositive_reference[
        ["Date", "ID", "region", "PM25FM",
         "PM25cfatm", "PM25cf1", "RH"]
    ].head(15)
)


Rows with PM25FM <= 0: 44


,Date,ID,region,PM25FM,PM25cfatm,PM25cf1,RH
1584,2018-11-24,CA10,West,-0.333333,1.238573,1.238573,50.124406
1636,2019-02-03,CA10,West,-0.333333,0.295137,0.295860,59.251866
1637,2019-02-04,CA10,West,-0.541667,0.288883,0.288883,56.799257
1649,2019-02-16,CA10,West,-0.166667,0.428765,0.428765,47.705603
2516,2018-07-29,CA15,West,-1.777778,6.138526,6.139611,72.959624
2517,2018-07-30,CA15,West,-1.772727,4.884192,4.884735,73.581244
2518,2018-07-31,CA15,West,-1.444444,5.968599,5.970164,71.269444
2519,2018-08-01,CA15,West,-0.083333,8.709523,8.718611,71.037963
2526,2018-08-08,CA15,West,-0.090909,10.073562,10.073627,69.316033
2696,2019-05-06,CA2,West,-0.130435,1.076477,1.076477,62.419444


In [17]:

# Summarize repeated sensor-date groups.
# This is diagnostic only; no records are removed or averaged.

sensor_measurement_cols = [
    "PM25cfatm",
    "PM25cf1",
    "RH",
    "TempC",
    "Dewpoint",
]

group_keys = ["ID", "Date", "region"]

repeated_rows = df_24hr.merge(
    repeated_sensor_dates[group_keys],
    on=group_keys,
    how="inner",
)

group_summary = (
    repeated_rows.groupby(group_keys, dropna=False)
    .agg(
        row_count=("PM25FM", "size"),
        reference_values=("PM25FM", "nunique"),
        monitor_categories=("FEMorFRM", "nunique"),
        monitor_types=("FEMorFRMtype", "nunique"),
        **{
            f"{col}_unique": (col, "nunique")
            for col in sensor_measurement_cols
        },
    )
    .reset_index()
)

print("Repeated sensor-date groups:", len(group_summary))

print("\nReference monitor category counts within groups:")
display(group_summary["monitor_categories"].value_counts().sort_index())

print("\nGroups with more than one monitor category:")
display(
    group_summary[group_summary["monitor_categories"] > 1].head(10)
)

print("\nGroups where sensor measurements vary:")
for col in sensor_measurement_cols:
    count = int((group_summary[f"{col}_unique"] > 1).sum())
    print(f"{col}: {count} groups")


Repeated sensor-date groups: 2199

Reference monitor category counts within groups:


monitor_categories
1    1053
2    1146
Name: count, dtype: int64


Groups with more than one monitor category:


,ID,Date,region,row_count,reference_values,monitor_categories,monitor_types,PM25cfatm_unique,PM25cf1_unique,RH_unique,TempC_unique,Dewpoint_unique
83,AZ4,2018-11-22,Central Southwest,2,2,2,2,1,1,1,1,1
84,AZ4,2018-12-04,Central Southwest,2,2,2,2,1,1,1,1,1
85,AZ4,2018-12-16,Central Southwest,2,2,2,2,1,1,1,1,1
86,AZ4,2018-12-28,Central Southwest,2,2,2,2,1,1,1,1,1
87,AZ4,2019-01-09,Central Southwest,2,2,2,2,1,1,1,1,1
88,AZ4,2019-01-21,Central Southwest,2,2,2,2,1,1,1,1,1
89,AZ4,2019-02-03,Central Southwest,2,2,2,2,1,1,1,1,1
90,AZ4,2019-02-14,Central Southwest,2,2,2,2,1,1,1,1,1
91,AZ4,2019-02-26,Central Southwest,2,2,2,2,1,1,1,1,1
92,AZ4,2019-03-10,Central Southwest,2,2,2,2,1,1,1,1,1



Groups where sensor measurements vary:
PM25cfatm: 0 groups
PM25cf1: 0 groups
RH: 0 groups
TempC: 0 groups
Dewpoint: 0 groups


In [18]:
# Investigate repeated FEM/FRM reference measurements

repeated_detail = (
    df_24hr[
        df_24hr.duplicated(
            subset=["ID", "Date", "region"],
            keep=False
        )
    ]
    .sort_values(["ID", "Date", "region", "FEMorFRM"])
)

display(
    repeated_detail[
        [
            "ID",
            "Date",
            "region",
            "FEMorFRM",
            "FEMorFRMtype",
            "PM25FM",
            "PM25cfatm",
            "PM25cf1"
        ]
    ].head(30)
)

,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cfatm,PM25cf1
37,AK1,2019-05-15,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,4.0,2.760172,2.760172
254,AK1,2019-05-15,Alaska,FRM,R & P Model 2000 PM-2.5 Air Sampler w/VSCC - G...,4.4,2.760172,2.760172
1,AK1,2019-05-18,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,3.8,3.359005,3.493916
255,AK1,2019-05-18,Alaska,FRM,R & P Model 2000 PM-2.5 Air Sampler w/VSCC - G...,4.0,3.359005,3.493916
2,AK1,2019-05-21,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,4.0,3.799601,3.799601
256,AK1,2019-05-21,Alaska,FRM,R & P Model 2000 PM-2.5 Air Sampler w/VSCC - G...,4.4,3.799601,3.799601
9,AK1,2019-05-24,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,2.7,2.696412,2.696412
257,AK1,2019-05-24,Alaska,FRM,R & P Model 2000 PM-2.5 Air Sampler w/VSCC - G...,2.6,2.696412,2.696412
23,AK1,2019-05-30,Alaska,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,3.9,2.550728,2.558361
259,AK1,2019-05-30,Alaska,FRM,R & P Model 2000 PM-2.5 Air Sampler w/VSCC - G...,3.7,2.550728,2.558361


In [42]:
# Summarize all repeated ID-Date-region groups

repeated_groups = (
    df_24hr[
        df_24hr.duplicated(
            subset=["ID", "Date", "region"],
            keep=False
        )
    ]
    .groupby(["ID", "Date", "region"])
    .agg(
        records=("PM25FM", "size"),
        monitor_categories=(
            "FEMorFRM",
            lambda x: ", ".join(sorted(x.dropna().unique()))
        ),
        instrument_types=(
            "FEMorFRMtype",
            "nunique"
        ),
        distinct_reference_values=("PM25FM", "nunique")
    )
    .reset_index()
)

# Count groups by their monitor-category composition
display(
    repeated_groups.groupby(
        ["records", "monitor_categories"],
        dropna=False
    )
    .size()
    .reset_index(name="group_count")
    .sort_values("group_count", ascending=False)
)

# Inspect groups with more than two records
display(
    repeated_groups[repeated_groups["records"] > 2].head(20)
)

,records,monitor_categories,group_count
0,2,FEM,942
1,2,"FEM, FRM",665
3,3,"FEM, FRM",438
2,2,FRM,111
4,4,"FEM, FRM",43


,ID,Date,region,records,monitor_categories,instrument_types,distinct_reference_values
652,GA1,2019-08-07,Southeast,3,"FEM, FRM",2,3
654,GA1,2019-08-10,Southeast,3,"FEM, FRM",2,3
657,GA1,2019-08-13,Southeast,3,"FEM, FRM",2,3
659,GA1,2019-08-16,Southeast,3,"FEM, FRM",2,2
662,GA1,2019-08-19,Southeast,3,"FEM, FRM",2,3
665,GA1,2019-08-22,Southeast,3,"FEM, FRM",2,3
668,GA1,2019-08-25,Southeast,3,"FEM, FRM",2,3
671,GA1,2019-08-28,Southeast,3,"FEM, FRM",2,2
674,GA1,2019-08-31,Southeast,3,"FEM, FRM",2,3
677,GA1,2019-09-12,Southeast,3,"FEM, FRM",2,2


In [43]:
# Count FEM and FRM records within each repeated group

monitor_counts = (
    df_24hr[
        df_24hr.duplicated(
            subset=["ID", "Date", "region"],
            keep=False
        )
    ]
    .groupby(["ID", "Date", "region", "FEMorFRM"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Summarize the combinations
display(
    monitor_counts.groupby(
        ["FEM", "FRM"],
        dropna=False
    )
    .size()
    .reset_index(name="group_count")
    .sort_values("group_count", ascending=False)
)

# Show examples where a category has multiple records
display(
    monitor_counts[
        (monitor_counts["FEM"] > 1) |
        (monitor_counts["FRM"] > 1)
    ].head(15)
)

,FEM,FRM,group_count
3,2,0,942
1,1,1,665
4,2,1,372
0,0,2,111
2,1,2,66
5,2,2,43


FEMorFRM,ID,Date,region,FEM,FRM
0,AK1,2019-05-15,Alaska,0,2
1,AK1,2019-05-18,Alaska,0,2
2,AK1,2019-05-21,Alaska,0,2
3,AK1,2019-05-24,Alaska,0,2
4,AK1,2019-05-30,Alaska,0,2
5,AK1,2019-06-02,Alaska,0,2
6,AK1,2019-06-05,Alaska,0,2
7,AK1,2019-06-08,Alaska,0,2
8,AK1,2019-06-11,Alaska,0,2
9,AK1,2019-06-14,Alaska,0,2


In [46]:
# Count reference instrument types by FEM/FRM category

instrument_summary = (
    df_24hr
    .groupby(["FEMorFRM", "FEMorFRMtype"], dropna=False)
    .agg(
        records=("PM25FM", "size"),
        distinct_sensors=("ID", "nunique"),
        distinct_reference_values=("PM25FM", "nunique")
    )
    .reset_index()
    .sort_values(
        ["FEMorFRM", "records"],
        ascending=[True, False]
    )
)

display(instrument_summary)

,FEMorFRM,FEMorFRMtype,records,distinct_sensors,distinct_reference_values
0,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,4746,18,972
2,FEM,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,2382,18,1532
6,FEM,Thermo Scientific TEOM 1405-DF Dichotomous FDM...,862,3,787
3,FEM,Teledyne T640X at 16.67 LPM - Broadband spectr...,760,4,683
1,FEM,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,718,2,389
4,FEM,Thermo Scientific 1405-F FDMS w/VSCC - FDMS Gr...,384,2,359
5,FEM,Thermo Scientific TEOM 1400 FDMS or 1405 8500C...,235,1,213
8,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,2255,22,252
7,FRM,R & P Model 2000 PM-2.5 Air Sampler w/VSCC - G...,88,3,61


In [47]:
# Inspect repeated groups containing multiple reference instrument types

instrument_comparison = (
    df_24hr[
        df_24hr.duplicated(
            subset=["ID", "Date", "region"],
            keep=False
        )
    ]
    .groupby(["ID", "Date", "region"])
    .agg(
        records=("PM25FM", "size"),
        instrument_types=("FEMorFRMtype", "nunique"),
        reference_min=("PM25FM", "min"),
        reference_max=("PM25FM", "max")
    )
    .reset_index()
)

instrument_comparison["reference_difference"] = (
    instrument_comparison["reference_max"] -
    instrument_comparison["reference_min"]
)

display(
    instrument_comparison[
        instrument_comparison["instrument_types"] > 1
    ].sort_values(
        "reference_difference",
        ascending=False
    ).head(15)
)

,ID,Date,region,records,instrument_types,reference_min,reference_max,reference_difference
1004,IA11,2019-07-04,North,3,2,22.370833,31.700000,9.329167
1576,IA8,2019-07-07,North,3,2,18.700000,27.458333,8.758333
1680,IA9,2019-07-07,North,3,2,18.700000,27.458333,8.758333
1007,IA11,2019-07-08,North,4,2,27.400000,36.154167,8.754167
1802,NC4,2018-12-07,Southeast,2,2,7.300000,15.854167,8.554167
1577,IA8,2019-07-08,North,3,2,22.900000,31.341667,8.441667
1681,IA9,2019-07-08,North,3,2,22.900000,31.341667,8.441667
1900,OK1,2019-09-15,Central Southwest,2,2,15.000000,23.275000,8.275000
782,IA1,2018-12-10,North,4,2,14.541667,22.200000,7.658333
1112,IA2,2018-12-10,North,4,2,14.541667,22.200000,7.658333


In [49]:
# Inspect the largest reference-measurement discrepancy

display(
    df_24hr[
        (df_24hr["ID"] == "IA11") &
        (df_24hr["Date"] == "2019-07-04")
    ][
        [
            "ID",
            "Date",
            "FEMorFRM",
            "FEMorFRMtype",
            "PM25FM",
            "PM25cfatm",
            "PM25cf1",
            "RH",
            "TempC"
        ]
    ]
)

,ID,Date,FEMorFRM,FEMorFRMtype,PM25FM,PM25cfatm,PM25cf1,RH,TempC
8609,IA11,2019-07-04,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,31.700000,26.351725,35.620547,56.962132,30.11454
8682,IA11,2019-07-04,FEM,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,22.370833,26.351725,35.620547,56.962132,30.11454
8957,IA11,2019-07-04,FEM,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,22.991667,26.351725,35.620547,56.962132,30.11454


In [50]:
# Compare FEM and FRM reference measurements within sensor-date groups

repeated_data = df_24hr[
    df_24hr.duplicated(
        subset=["ID", "Date", "region"],
        keep=False
    )
].copy()

reference_comparison = (
    repeated_data
    .groupby(["ID", "Date", "region", "FEMorFRM"])["PM25FM"]
    .mean()
    .unstack()
    .reset_index()
)

# Keep only groups containing both FEM and FRM
both_available = reference_comparison.dropna(
    subset=["FEM", "FRM"]
).copy()

both_available["FRM_minus_FEM"] = (
    both_available["FRM"] - both_available["FEM"]
)

display(
    both_available["FRM_minus_FEM"].describe()
)

print("Sensor-date groups with both FEM and FRM:",
      len(both_available))

print("FEM higher than FRM:",
      (both_available["FRM_minus_FEM"] < 0).sum())

print("FRM higher than FEM:",
      (both_available["FRM_minus_FEM"] > 0).sum())

print("Equal FEM and FRM:",
      (both_available["FRM_minus_FEM"] == 0).sum())

count    1146.000000
mean       -0.609174
std         2.031943
min        -8.554167
25%        -1.885417
50%        -0.583333
75%         0.495994
max         9.018750
Name: FRM_minus_FEM, dtype: float64

Sensor-date groups with both FEM and FRM: 1146
FEM higher than FRM: 728
FRM higher than FEM: 416
Equal FEM and FRM: 2


In [51]:
# Examine FEM-FRM differences by PurpleAir sensor

sensor_difference = (
    both_available
    .groupby("ID")
    .agg(
        paired_dates=("FRM_minus_FEM", "size"),
        mean_difference=("FRM_minus_FEM", "mean"),
        median_difference=("FRM_minus_FEM", "median"),
        std_difference=("FRM_minus_FEM", "std")
    )
    .reset_index()
    .sort_values("paired_dates", ascending=False)
)

display(sensor_difference)

,ID,paired_dates,mean_difference,median_difference,std_difference
7,IA3,171,0.363461,0.381818,1.263418
11,NC4,158,-0.713221,-0.531250,1.693284
6,IA2,131,0.237623,0.066667,2.098169
3,IA1,125,0.677621,0.516667,2.113124
5,IA11,90,-1.667661,-1.928125,2.324269
10,IA9,87,-2.461985,-2.250000,1.615281
9,IA8,82,-2.307238,-2.280209,1.854634
2,GA1,69,-0.305968,-0.400000,0.959650
8,IA4,53,0.015208,0.008333,1.148753
4,IA10,45,-2.118041,-2.375000,1.797030


In [52]:
# Check variation among repeated measurements within FEM or FRM

within_category = (
    df_24hr[
        df_24hr.duplicated(
            subset=["ID", "Date", "region"],
            keep=False
        )
    ]
    .groupby(["ID", "Date", "region", "FEMorFRM"])
    .agg(
        records=("PM25FM", "size"),
        instrument_types=("FEMorFRMtype", "nunique"),
        reference_min=("PM25FM", "min"),
        reference_max=("PM25FM", "max")
    )
    .reset_index()
)

within_category["reference_difference"] = (
    within_category["reference_max"] -
    within_category["reference_min"]
)

multiple_measurements = within_category[
    within_category["records"] > 1
]

print("Groups with multiple measurements in the same category:",
      len(multiple_measurements))

print("Groups with different reference values:",
      (multiple_measurements["reference_difference"] > 0).sum())

print("Groups with identical reference values:",
      (multiple_measurements["reference_difference"] == 0).sum())

display(
    multiple_measurements.sort_values(
        "reference_difference",
        ascending=False
    ).head(15)
)

Groups with multiple measurements in the same category: 1577
Groups with different reference values: 1533
Groups with identical reference values: 44


,ID,Date,region,FEMorFRM,records,instrument_types,reference_min,reference_max,reference_difference
285,CA13,2018-08-07,West,FEM,2,1,18.652174,25.304348,6.652174
1492,IA2,2018-08-08,North,FEM,2,1,4.913043,10.625000,5.711957
1519,IA2,2018-09-01,North,FEM,2,1,4.875000,10.125000,5.250000
1818,IA3,2018-02-13,North,FEM,2,1,11.250000,16.458333,5.208333
1820,IA3,2018-02-14,North,FEM,2,1,11.500000,16.708333,5.208333
234,CA13,2018-06-16,West,FEM,2,1,9.375000,14.458333,5.083333
1494,IA2,2018-08-09,North,FEM,2,1,9.750000,14.833333,5.083333
169,CA13,2018-04-12,West,FEM,2,1,5.250000,10.125000,4.875000
1850,IA3,2018-04-11,North,FEM,2,1,17.250000,22.000000,4.750000
1764,IA3,2017-11-04,North,FEM,2,1,15.500000,20.125000,4.625000


In [53]:
# Inspect every available field for the repeated CA13 records

example_records = df_24hr[
    (df_24hr["ID"] == "CA13") &
    (df_24hr["Date"] == "2018-08-07")
]

display(example_records.T)

,2059,2237
Unnamed: 0,2060,2238
Date,2018-08-07,2018-08-07
ID,CA13,CA13
region,West,West
FEMorFRM,FEM,FEM
FEMorFRMtype,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...
PM25FM,25.304348,18.652174
PM25cfatm,28.327371,28.327371
PM25cf1,32.725077,32.725077
b0.3um,3967.54259,3967.54259


In [54]:
# Compare possible Dataset A analysis populations

print("Original rows:", len(df_24hr))

print(
    "Unique sensor-date-region combinations:",
    df_24hr[["ID", "Date", "region"]].drop_duplicates().shape[0]
)

# Approach 1: Keep all reference records
print("Approach 1 - Keep all records:", len(df_24hr))

# Approach 2: Average all reference measurements per sensor-date
all_reference_average = (
    df_24hr
    .groupby(["ID", "Date", "region"], as_index=False)
    .agg(
        PM25FM=("PM25FM", "mean"),
        PM25cfatm=("PM25cfatm", "first"),
        PM25cf1=("PM25cf1", "first")
    )
)

print(
    "Approach 2 - Average all reference records:",
    len(all_reference_average)
)

# Approach 3: Keep FEM only, average within sensor-date
fem_only = df_24hr[df_24hr["FEMorFRM"] == "FEM"]

fem_average = (
    fem_only
    .groupby(["ID", "Date", "region"], as_index=False)
    .agg(
        PM25FM=("PM25FM", "mean"),
        PM25cfatm=("PM25cfatm", "first"),
        PM25cf1=("PM25cf1", "first")
    )
)

print(
    "Approach 3 - FEM only, averaged per sensor-date:",
    len(fem_average)
)

Original rows: 12430
Unique sensor-date-region combinations: 9707
Approach 1 - Keep all records: 12430
Approach 2 - Average all reference records: 9707
Approach 3 - FEM only, averaged per sensor-date: 8730


In [55]:
# Classify every sensor-date group by available reference category

reference_coverage = (
    df_24hr
    .groupby(["ID", "Date", "region"])
    .agg(
        records=("PM25FM", "size"),
        FEM_records=("FEMorFRM", lambda x: (x == "FEM").sum()),
        FRM_records=("FEMorFRM", lambda x: (x == "FRM").sum())
    )
    .reset_index()
)

reference_coverage["reference_availability"] = reference_coverage.apply(
    lambda row: (
        "Both FEM and FRM"
        if row["FEM_records"] > 0 and row["FRM_records"] > 0
        else "FEM only"
        if row["FEM_records"] > 0
        else "FRM only"
    ),
    axis=1
)

display(
    reference_coverage["reference_availability"]
    .value_counts()
    .reset_index()
)

print("Total unique sensor-date groups:",
      len(reference_coverage))

print("Groups with multiple records:",
      (reference_coverage["records"] > 1).sum())

,reference_availability,count
0,FEM only,7584
1,Both FEM and FRM,1146
2,FRM only,977


Total unique sensor-date groups: 9707
Groups with multiple records: 2199


In [56]:
# Inspect nonpositive reference measurements

nonpositive_ref = df_24hr[df_24hr["PM25FM"] <= 0]

print("Total nonpositive reference values:", len(nonpositive_ref))

display(
    nonpositive_ref[
        [
            "ID",
            "Date",
            "region",
            "FEMorFRM",
            "FEMorFRMtype",
            "PM25FM",
            "PM25cfatm",
            "PM25cf1"
        ]
    ].sort_values("PM25FM")
)

Total nonpositive reference values: 44


,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cfatm,PM25cf1
2516,CA15,2018-07-29,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.777778,6.138526,6.139611
2517,CA15,2018-07-30,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.772727,4.884192,4.884735
3738,CA4,2019-08-03,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.625000,2.259799,2.283375
3031,CA2,2018-03-21,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.500000,1.070726,1.070726
2518,CA15,2018-07-31,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.444444,5.968599,5.970164
3023,CA2,2018-03-13,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.380952,1.017975,1.018309
2960,CA2,2018-01-09,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.130435,0.654005,0.654005
2959,CA2,2018-01-08,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.083333,4.863587,4.866857
3032,CA2,2018-03-22,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.045455,0.778960,0.778960
3087,CA2,2018-05-16,West,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.000000,1.192708,1.192708


In [57]:
# Check alternative reference measurements for nonpositive records

nonpositive_keys = nonpositive_ref[
    ["ID", "Date", "region"]
].drop_duplicates()

alternative_records = df_24hr.merge(
    nonpositive_keys,
    on=["ID", "Date", "region"],
    how="inner"
)

display(
    alternative_records[
        [
            "ID",
            "Date",
            "FEMorFRM",
            "FEMorFRMtype",
            "PM25FM"
        ]
    ].sort_values(["ID", "Date", "FEMorFRM"])
)

print("Unique affected sensor-date groups:",
      len(nonpositive_keys))

print("Total records across those groups:",
      len(alternative_records))

print("Positive alternative reference records:",
      (
          (alternative_records["PM25FM"] > 0) &
          (alternative_records["FEMorFRM"] == "FRM")
      ).sum())

,ID,Date,FEMorFRM,FEMorFRMtype,PM25FM
0,CA10,2018-11-24,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-0.333333
1,CA10,2019-02-03,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-0.333333
2,CA10,2019-02-04,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-0.541667
3,CA10,2019-02-16,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-0.166667
4,CA15,2018-07-29,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.777778
5,CA15,2018-07-30,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.772727
6,CA15,2018-07-31,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.444444
7,CA15,2018-08-01,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-0.083333
8,CA15,2018-08-08,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-0.090909
11,CA16,2018-05-16,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,-1.000000


Unique affected sensor-date groups: 44
Total records across those groups: 46
Positive alternative reference records: 2


In [58]:
# Check missingness in environmental predictors

environment_missing = pd.DataFrame({
    "Missing count": df_24hr[["RH", "TempC", "Dewpoint"]].isna().sum(),
    "Missing percentage": (
        df_24hr[["RH", "TempC", "Dewpoint"]].isna().mean() * 100
    )
})

display(environment_missing)

# Check whether missing temperature and humidity occur together
print(
    "Rows missing both RH and TempC:",
    df_24hr[["RH", "TempC"]].isna().all(axis=1).sum()
)

print(
    "Rows missing TempC only:",
    (df_24hr["TempC"].isna() & df_24hr["RH"].notna()).sum()
)

print(
    "Rows missing RH only:",
    (df_24hr["RH"].isna() & df_24hr["TempC"].notna()).sum()
)

,Missing count,Missing percentage
RH,17,0.136766
TempC,184,1.480290
Dewpoint,184,1.480290


Rows missing both RH and TempC: 17
Rows missing TempC only: 167
Rows missing RH only: 0


In [59]:
# Check environmental missingness by region

region_missing = (
    df_24hr.groupby("region")
    .agg(
        Total_rows=("ID", "size"),
        Missing_RH=("RH", lambda x: x.isna().sum()),
        Missing_TempC=("TempC", lambda x: x.isna().sum()),
        Missing_Dewpoint=("Dewpoint", lambda x: x.isna().sum())
    )
)

region_missing["Missing_TempC_pct"] = (
    region_missing["Missing_TempC"] /
    region_missing["Total_rows"] * 100
)

display(region_missing.round(2))

,Total_rows,Missing_RH,Missing_TempC,Missing_Dewpoint,Missing_TempC_pct
region,,,,,
Alaska,369,0,0,0,0.00
Central Southwest,1504,2,2,2,0.13
North,5762,3,170,170,2.95
Southeast,1033,1,1,1,0.10
West,3762,11,11,11,0.29


In [60]:
# Identify sensors with missing temperature values

sensor_missing = (
    df_24hr.groupby(["region", "ID"])
    .agg(
        Total_rows=("TempC", "size"),
        Missing_TempC=("TempC", lambda x: x.isna().sum())
    )
)

sensor_missing = sensor_missing[
    sensor_missing["Missing_TempC"] > 0
].copy()

sensor_missing["Missing_TempC_pct"] = (
    sensor_missing["Missing_TempC"] /
    sensor_missing["Total_rows"] * 100
)

display(
    sensor_missing.sort_values(
        "Missing_TempC",
        ascending=False
    ).round(2)
)

Total_rows  Missing_TempC  Missing_TempC_pct
region            ID                                                
North             WI4          185            167              90.27
West              CA18         398              4               1.01
                  CA4          282              4               1.42
North             WA1          561              3               0.53
West              CA3          265              2               0.75
Central Southwest AZ4          435              1               0.23
                  KS2          111              1               0.90
Southeast         NC4          700              1               0.14
West              CA5          246              1               0.41

In [61]:
# Inspect WI4 temperature availability over time

wi4 = df_24hr[df_24hr["ID"] == "WI4"].copy()
wi4["Date"] = pd.to_datetime(wi4["Date"])

wi4["YearMonth"] = wi4["Date"].dt.to_period("M")

wi4_monthly = (
    wi4.groupby("YearMonth")
    .agg(
        Total_rows=("TempC", "size"),
        Missing_TempC=("TempC", lambda x: x.isna().sum())
    )
)

wi4_monthly["Missing_TempC_pct"] = (
    wi4_monthly["Missing_TempC"] /
    wi4_monthly["Total_rows"] * 100
)

display(wi4_monthly)

,Total_rows,Missing_TempC,Missing_TempC_pct
YearMonth,,,
2019-01,10,0,0.000000
2019-02,2,0,0.000000
2019-03,4,0,0.000000
2019-05,1,1,100.000000
2019-06,35,35,100.000000
2019-07,33,31,93.939394
2019-08,37,37,100.000000
2019-09,35,35,100.000000
2019-10,28,28,100.000000


In [62]:
# Check existing corrected values in rows with missing temperature

missing_temp = df_24hr[df_24hr["TempC"].isna()]

display(
    missing_temp[
        ["PM25cfatm", "PM25cf1", "PM25PAlinearcor",
         "PM25PARHcor", "RH", "TempC", "Dewpoint"]
    ].isna().sum()
)

PM25cfatm            0
PM25cf1              0
PM25PAlinearcor      0
PM25PARHcor         17
RH                  17
TempC              184
Dewpoint           184
dtype: int64

### 6.5 Data Profile Findings — 24-Hour Collocation Dataset

#### Source and Structure

The dataset was downloaded from the EPA Pasteur repository on October 22, 2026. The main file, `Full24hrdataset.csv`, contains **12,430 rows and 31 columns**.

The accompanying data dictionary identifies this file as a 24-hour averaged dataset used in the published development of a United States-wide PurpleAir PM2.5 correction.

#### Observation Period and Identifiers

The observation dates range from September 30, 2017, through January 13, 2020. The dataset contains 830 distinct dates, with no unparseable dates or observations outside the expected 2017–2020 period.

There are 49 distinct `ID` values distributed across five regions: North, West, Central Southwest, Southeast, and Alaska. No `ID` was associated with more than one region in this dataset.

The dictionary describes `ID` as a state-number identifier. Therefore, it should not automatically be interpreted as a unique physical sensor serial number.

#### Key Variables

The data dictionary defines the principal variables as follows:

- `PM25FM`: PM2.5 measurement from a Federal Equivalent Method (FEM) or Federal Reference Method (FRM) monitor, in µg/m³.
- `PM25cfatm`: PurpleAir AB-channel-averaged lower CF PM2.5 measurement, in µg/m³.
- `PM25cf1`: PurpleAir AB-channel-averaged higher CF PM2.5 measurement, in µg/m³.
- `RH`: Relative humidity measured by PurpleAir, in percent.
- `TempC`: Temperature measured by PurpleAir, in °C.
- `Dewpoint`: Dewpoint calculated from PurpleAir temperature and relative humidity.
- `FEMorFRM` and `FEMorFRMtype`: Reference-monitor category and monitor type.

The file also includes previously calculated correction fields, ratios, and differences. These fields will be distinguished from models independently trained and evaluated in this project.

#### Uniqueness and Repeated Observations

No exact duplicate rows were identified. However, 2,199 distinct `ID`-`Date`-`region` combinations occur more than once, representing 4,922 rows and 2,723 rows beyond one record per combination.

Among these repeated groups, 1,053 contain only one distinct `FEMorFRM` category, while 1,146 contain both FEM and FRM categories. In the inspected examples, PurpleAir measurements and environmental variables are identical across repeated records, whereas reference measurements and monitor types differ.

Further investigation found that repeated groups can contain two, three, or four reference records. The repeated-record structure is not limited to one FEM and one FRM measurement. Some groups contain multiple readings within the same reference category.

Across the full dataset, there are 9,707 unique sensor-date-region groups:
- 7,584 groups have FEM measurements only.
- 1,146 groups have both FEM and FRM measurements.
- 977 groups have FRM measurements only.

In groups containing both categories, FEM and FRM measurements do not consistently agree or show one category being higher. Their differences vary across sensor-date groups. Repeated measurements within the same category can also differ.

The available columns do not identify a unique physical reference-monitor ID for every measurement. Consequently, the precise reason for every within-category repeated measurement cannot be established from the available fields.

**Decision:** All original records will be preserved during profiling. Repeated reference measurements will not be automatically deleted or averaged. A FEM-priority reference-selection approach is under consideration because FEM measurements are available for 8,730 unique groups, approximately 90% of all groups. The 977 FRM-only groups will be retained for separate coverage and sensitivity analysis. The final reference-selection rule remains provisional pending review of the documentation and analytical requirements.

#### Missingness

The initial missing-value assessment identified the following missing values:

| Variable | Missing values | Missing percentage |
|---|---:|---:|
| `TempC` | 184 | 1.48% |
| `Dewpoint` | 184 | 1.48% |
| `RH` | 17 | 0.14% |
| `PM25PARHcor` | 17 | 0.14% |
| `ratioPARHcor_FM` | 166 | 1.34% |
| `ratiocfatm_FM` | 149 | 1.20% |
| `ratiocf1_FM` | 149 | 1.20% |

The data dictionary states that `ratioPARHcor_FM` is set to missing when the FEM/FRM measurement is below 1 µg/m³. This provides a documented reason for at least some missing ratio values. The complete causes of missingness in the ratio fields have not yet been established.

Further investigation of the environmental variables found:

- All 17 rows missing RH also lack TempC and Dewpoint.
- An additional 167 rows are missing TempC and Dewpoint while RH remains available.
- Therefore, 167 of the 184 rows with missing temperature still have humidity measurements.

Missing temperature and dewpoint values are concentrated in the North region. Of the 184 affected rows, 170 (92.4%) are from the North. The remaining missing temperature records are distributed across Central Southwest, Southeast, and West. Alaska has no missing temperature records.

Sensor-level investigation showed that the missingness is particularly concentrated in sensor WI4 in the North region. WI4 has 167 missing temperature records out of 185 observations (90.27%), accounting for approximately 90.8% of all missing temperature records in the dataset.

A monthly examination of WI4 showed that temperature is available for its January–March 2019 observations but missing for nearly all observations from May through October 2019. This indicates a sensor-period-specific data availability issue rather than uniformly distributed missingness.

An additional check of existing measurement and correction fields found that all 184 rows with missing temperature retain `PM25cfatm`, `PM25cf1`, and `PM25PAlinearcor`. The existing `PM25PARHcor` value is available for 167 of these rows and missing for the 17 rows that also lack RH.

**Decision:** Original records will be retained, and no imputation will be performed during profiling. Missing temperature and dewpoint values will be documented and considered when selecting observations for temperature-dependent analyses. The 167 observations with available RH can still be considered for appropriate humidity-related analyses. WI4 and its May–October 2019 period will be flagged for sensitivity analysis. Existing correction values will be retained as supplied and will not be recalculated or treated as independently trained model predictions.

#### Measurement Values Requiring Investigation

The reference measurement `PM25FM` contains 44 non-positive values: one zero and 43 negative values. These values represent 44 unique sensor-date groups. All 44 non-positive readings are FEM measurements from the Met One BAM-1020 instrument type.

An examination of alternative reference records found 46 total records across the 44 affected groups. Only two groups contain an alternative positive FRM measurement:
- IA1, May 19, 2019: FEM = -0.583333 µg/m³; FRM = 1.1 µg/m³.
- IA2, May 20, 2019: FEM = -0.416667 µg/m³; FRM = 1.8 µg/m³.

The remaining 42 affected groups do not have a positive alternative reference measurement in the available records.

**Decision:** All original measurements will be retained and flagged. Non-positive reference values will not be silently replaced with alternative FRM measurements or automatically modified. Their eligibility for the primary calibration evaluation will be determined by the analysis-specific metric and reference-selection rules. Sensitivity analysis will be considered to assess their influence.

#### Documentation Discrepancy: `diffPARHcor_FM`

The data dictionary describes `diffPARHcor_FM` as the difference between the FEM/FRM reference measurement and the RH-corrected PurpleAir measurement. However, an inspected example appears to follow the opposite sign convention.

For the inspected example:

- `PM25PARHcor` = 15.132007 µg/m³
- `PM25FM` = 9.5 µg/m³
- `PM25PARHcor - PM25FM` = +5.632007 µg/m³

The observed value is consistent with:

`diffPARHcor_FM = PM25PARHcor - PM25FM`

rather than the reference-minus-corrected-PurpleAir convention suggested by the dictionary's description.

**Decision:** The supplied difference field will not be used for model evaluation until its sign convention is verified. For this project, residuals will be calculated explicitly as:

`Predicted PM2.5 - Reference PM2.5`

This ensures that positive and negative errors have a consistent interpretation across independently developed models. Reversing the sign would change the interpretation of bias direction but would not change absolute-error metrics such as MAE.

#### Preliminary Data-Quality Decisions

| Issue | Decision | Reason |
|---|---|---|
| Exact duplicate rows | No removal required | No exact duplicates were identified. |
| Repeated `ID`-`Date`-`region` combinations | Preserve and flag | Repeated records represent different reference measurements, not exact duplicates. |
| FEM and FRM differences | Keep categories distinct | Measurements can differ within the same sensor-date group. |
| Reference-monitor selection | FEM-priority under consideration | FEM provides approximately 90% unique-group coverage; FRM-only records require sensitivity consideration. |
| Non-positive `PM25FM` values | Retain and flag | All 44 are FEM readings; only two groups have positive alternative FRM readings. |
| Missing RH, TempC, and Dewpoint | Retain; no profiling-stage imputation | Missingness is limited but concentrated in WI4 and a specific period. |
| Existing correction fields | Preserve as supplied | These are published correction outputs, not predictions from the project's own models. |
| Missing ratio fields | Document and investigate as needed | Some missingness is explained by the data dictionary, but not all causes are established. |
| `diffPARHcor_FM` | Avoid until sign convention is verified | The inspected example appears inconsistent with the dictionary description. |

#### Implications for the Project

Dataset A is suitable for further exploratory analysis, but its repeated reference-monitor structure must be accounted for when defining the analytical observation unit and selecting the reference target.

The primary analytical approach is likely to prioritize FEM measurements while preserving FRM-only observations for separate coverage and sensitivity analysis. This remains a provisional choice rather than a finalized rule.

The repeated records also have implications for model evaluation. Observations sharing the same underlying sensor-date and PurpleAir predictors must not be allowed to leak across training and testing partitions. The eventual split strategy must group related observations appropriately.

The published correction fields will not be treated as outputs from newly trained models. Independent calibration models will use clearly defined predictors, an appropriate reference target, and a documented holdout strategy.

Missing environmental values, non-positive reference readings, and the WI4 monitoring-period issue will be considered when defining model-specific analytical samples and sensitivity checks.

#### Remaining Decisions Before Modeling

The main profiling findings have been investigated, but the following analytical decisions still need to be finalized after Dataset B profiling:

1. Select and document the primary reference-monitor strategy for Dataset A, including how FRM-only groups will be handled.
2. Define the treatment of non-positive reference measurements for the primary evaluation and sensitivity analysis.
3. Decide how missing environmental predictors will be handled for each model.
4. Define a grouped train/test strategy that prevents repeated sensor-date observations from appearing in both partitions.
5. Verify or avoid the supplied `diffPARHcor_FM` field and consistently calculate project-specific residuals.

These decisions will be finalized before model development. The dataset's original records and supplied correction values will remain preserved.


## Dataset B: Hourly PurpleAir Wildfire-Smoke Dataset

### Purpose

This dataset will help us investigate how well low-cost PurpleAir PM2.5 measurements agree with reference-monitor measurements during the documented smoke-related observation period. It will support our research question about whether calibration improves agreement under wildfire-smoke conditions.

### Profiling approach

Before modeling, we will inspect the dataset's structure, time coverage, missing values, repeated observations, sensor and city coverage, measurement ranges, and potential data-quality problems. We will consult the supplied README and supporting documentation where available.

We will not assume that every row is unique, every measurement is valid, or that the supplied `adj` values can be used as our independently trained model predictions. Any decisions about exclusions, corrections, and train/test splitting will be documented separately.

**Important:** These are exploratory data-quality checks. We will not select a final model or tune it using test-set results during profiling.


In [19]:


current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook "
        "from inside the low-cost-pm25-calibration repository."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SMOKE_DIR = RAW_DIR / "hourly_smoke"

SMOKE_FILE = (
    SMOKE_DIR / "Finaldataset_correcteddatasetwholding_10_26_22.csv"
)

README_FILE = SMOKE_DIR / "Readme_PAsmoke_1_18_22.docx"

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", SMOKE_DIR)
print("Main CSV exists:", SMOKE_FILE.is_file())
print("README exists:", README_FILE.is_file())


Project root: .
Dataset directory: data/raw/hourly_smoke
Main CSV exists: True
README exists: True


In [20]:

if not SMOKE_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {SMOKE_DIR}")

file_inventory = []

for path in sorted(SMOKE_DIR.iterdir()):
    if path.is_file():
        file_inventory.append({
            "file": path.name,
            "size_MB": round(path.stat().st_size / (1024 ** 2), 3),
            "extension": path.suffix.lower() or "(no extension)",
        })

file_inventory_df = pd.DataFrame(file_inventory)

display(file_inventory_df)

if not SMOKE_FILE.is_file():
    raise FileNotFoundError(
        f"Main dataset CSV is missing: {SMOKE_FILE}"
    )


,file,size_MB,extension
0,AQS3yearT640compare.csv,12.911,.csv
1,AQS3yearT640xcompare.csv,7.078,.csv
2,Fig4_ForksSalmon_cf1atm3_15_23.csv,0.028,.csv
3,Fig5_Nilson_corrected.csv,0.003,.csv
4,FigS7_Nilsonfits.csv,13.426,.csv
5,Finaldataset_correcteddatasetwholding_10_26_22...,249.834,.csv
6,Readme_PAsmoke_1_18_22.docx,0.024,.docx
7,nowcasted_dataset_20220707b.csv,49.874,.csv
8,nowcasted_dataset_RH_20220712.csv,27.180,.csv


In [21]:

if README_FILE.is_file():
    try:
        from docx import Document

        doc = Document(README_FILE)

        print("README PARAGRAPHS")
        for paragraph in doc.paragraphs:
            text = paragraph.text.strip()
            if text:
                print("-", text)

        print("\nREADME TABLES")
        for table_number, table in enumerate(doc.tables, start=1):
            print(f"\nTable {table_number}")
            for row in table.rows:
                print(" | ".join(cell.text.strip() for cell in row.cells))

    except ImportError:
        print(
            "python-docx is not installed. Install it with "
            "`pip install python-docx`, then rerun this cell."
        )
else:
    print("README file not found at the expected path.")
    print("Use the file inventory above to identify the available documentation.")


README PARAGRAPHS
- For paper: Correction and Accuracy of PurpleAir PM2.5 Measurements for Extreme Wildfire Smoke
- Karoline K. Barkjohn, Amara L. Holder, Samuel G. Frederick, Andrea L. Clements
- Contact:
- Barkjohn.Karoline@epa.gov
- Files
- Finaldataset_correcteddatasetwholding_10_26_22.csv
- This dataset includes all the data corrected using withholding. So a correction was built on all but one site or week of the year and then applied to that withheld site or year
- nowcasted_dataset_RH_20220712.csv
- This dataset does not use withholding it just uses the quadradic fit reported in the paper and the reported transition zone 570-611 along with the original US-wide correction to correct the data
- nowcasted_dataset_20220707b.csv
- This dataset uses the PurpleAir corrected data from US-wide to quadradic transitioning from 570-611 (raw PAcf_1 ug/m3). Where each quadradic fit is built using withholding
- Fig4_ForksSalmon.csv
- Fig5_Nilson_corrected.csv
- Mock dataset showing the differe

In [22]:

df_smoke = pd.read_csv(SMOKE_FILE, low_memory=False)

print(f"Rows: {df_smoke.shape[0]:,}")
print(f"Columns: {df_smoke.shape[1]:,}")
print(f"Approximate in-memory size: {df_smoke.memory_usage(deep=True).sum() / (1024**2):,.2f} MB")

print("\nColumn names:")
print(df_smoke.columns.tolist())


Rows: 1,513,776
Columns: 13
Approximate in-memory size: 689.64 MB

Column names:
['Unnamed: 0', 'hr', 'city', 'id', 'RH', 'PA', 'ref', 'monitortype', 'State', 'week', 'adj', 'adjtype', 'wholdtype']


In [23]:

print("DATA TYPES")
display(
    df_smoke.dtypes.astype(str)
    .rename("dtype")
    .to_frame()
)

print("\nFIRST 10 ROWS")
display(df_smoke.head(10))

print("\nRANDOM SAMPLE OF 10 ROWS")
display(df_smoke.sample(n=min(10, len(df_smoke)), random_state=42))

print("\nNON-NULL VALUE COUNTS")
display(df_smoke.count().rename("non_null_count").to_frame())


DATA TYPES


,dtype
Unnamed: 0,int64
hr,str
city,str
id,str
RH,float64
PA,float64
ref,float64
monitortype,str
State,str
week,float64



FIRST 10 ROWS


,Unnamed: 0,hr,city,id,RH,PA,ref,monitortype,State,week,adj,adjtype,wholdtype
0,1,2018-12-31 00:00:00,Atascadero,Atascadero,65.800000,43.397725,22.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,20.402327,C,LOBD
1,2,2018-12-31 00:00:00,Cedar Rapids,PALinn2,54.644442,13.762445,7.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,8.861555,C,LOBD
2,3,2018-12-31 00:00:00,Marysville,PAMarysville 7th,80.422221,50.088111,22.0,Thermo Scientific 1405-F FDMS w/VSCC - FDMS Gr...,NaN,53.0,23.135010,C,LOBD
3,4,2018-12-31 01:00:00,Atascadero,Atascadero,65.200000,30.376135,11.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,15.217768,C,LOBD
4,5,2018-12-31 01:00:00,Cedar Rapids,PALinn2,52.711111,14.149000,6.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,9.006140,C,LOBD
5,6,2018-12-31 01:00:00,Marysville,PAMarysville 7th,80.833332,23.814880,7.9,Thermo Scientific 1405-F FDMS w/VSCC - FDMS Gr...,NaN,53.0,12.672775,C,LOBD
6,7,2018-12-31 02:00:00,Atascadero,Atascadero,61.230770,27.511955,12.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,14.101239,C,LOBD
7,8,2018-12-31 02:00:00,Cedar Rapids,PALinn2,50.666668,13.378666,9.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,8.718166,C,LOBD
8,9,2018-12-31 02:00:00,Marysville,PAMarysville 7th,81.266668,29.492666,5.7,Thermo Scientific 1405-F FDMS w/VSCC - FDMS Gr...,NaN,53.0,14.872451,C,LOBD
9,10,2018-12-31 03:00:00,Atascadero,Atascadero,59.692310,28.043460,12.0,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0,14.307782,C,LOBD



RANDOM SAMPLE OF 10 ROWS


,Unnamed: 0,hr,city,id,RH,PA,ref,monitortype,State,week,adj,adjtype,wholdtype
647185,773334,2019-07-17 07:00:00,Marysville,PAMarysville 7th,72.500000,3.199891,2.00,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,28.0,1.868815,F,LOBD
792610,918759,2020-08-23 20:00:00,Edmond,LTPP_OK,36.966667,18.357500,18.90,Teledyne T640X at 16.67 LPM - Broadband spectr...,LTPP_OK,NaN,10.440033,C,LOSO
539825,665974,2019-05-19 02:00:00,Appleton,PAAppleton,80.000000,12.292969,5.55,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,NaN,19.0,5.295516,A,LOBD
1364134,1616431,2020-03-23 05:00:00,Cedar Rapids,PALinn2,59.633332,24.150332,9.00,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,PALinn2,NaN,13.264381,A,LOSO
1435273,1687570,2019-11-03 21:00:00,Wilmington,LTPP_DE,46.566667,26.062500,7.20,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,LTPP_DE,NaN,20.325798,F,LOSO
554516,680665,2019-03-26 20:00:00,Atascadero,Atascadero,67.422220,1.431334,3.00,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,12.0,0.688223,A,LOBD
822912,949061,2020-01-01 19:00:00,Topeka,PAKNI_E2,45.733333,2.279500,4.60,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,PAKNI_E2,NaN,4.582257,C,LOSO
15337,15338,2020-07-22 16:00:00,Phoenix,LTPP_AZ,16.466667,1.935167,6.60,Thermo Scientific TEOM 1405-DF Dichotomous FDM...,NaN,29.0,4.517850,C,LOBD
248074,248075,2019-09-01 17:00:00,Wilmington,LTPP_DE,49.133332,3.855167,5.00,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,NaN,34.0,4.934091,D,LOBD
1381357,1633654,2019-04-16 16:00:00,Appleton,PAAppleton,47.822221,19.790334,12.30,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,PAAppleton,NaN,11.997860,A,LOSO



NON-NULL VALUE COUNTS


,non_null_count
Unnamed: 0,1513776
hr,1513776
city,1513776
id,1513776
RH,1513776
PA,1513776
ref,1513776
monitortype,1513776
State,756888
week,756888


In [24]:

expected_columns = {"hr", "city", "id", "RH", "PA", "ref", "adj"}

actual_columns = set(df_smoke.columns)

print("Expected key columns present:")
for column in sorted(expected_columns):
    print(f"  {column}: {column in actual_columns}")

print("\nMissing expected columns:", sorted(expected_columns - actual_columns))
print("Additional columns:", sorted(actual_columns - expected_columns))

if not expected_columns.issubset(actual_columns):
    print(
        "\nPlease stop and share this output before running later cells. "
        "The dataset schema may differ from the expected version."
    )


Expected key columns present:
  PA: True
  RH: True
  adj: True
  city: True
  hr: True
  id: True
  ref: True

Missing expected columns: []
Additional columns: ['State', 'Unnamed: 0', 'adjtype', 'monitortype', 'week', 'wholdtype']


In [25]:

missing_summary = pd.DataFrame({
    "missing_count": df_smoke.isna().sum(),
    "missing_percent": (df_smoke.isna().mean() * 100).round(4),
    "non_missing_count": df_smoke.notna().sum(),
})

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False,
)

display(missing_summary)


,missing_count,missing_percent,non_missing_count
State,756888,50.0,756888
week,756888,50.0,756888
Unnamed: 0,0,0.0,1513776
hr,0,0.0,1513776
city,0,0.0,1513776
id,0,0.0,1513776
RH,0,0.0,1513776
PA,0,0.0,1513776
ref,0,0.0,1513776
monitortype,0,0.0,1513776


In [26]:

print("UNIQUE COUNTS")
for column in ["city", "id", "hr"]:
    print(f"{column}: {df_smoke[column].nunique(dropna=True):,} distinct non-missing values")
    print(f"  Missing: {df_smoke[column].isna().sum():,}")

print("\nTOP 20 CITIES")
display(df_smoke["city"].value_counts(dropna=False).head(20).to_frame("rows"))

print("\nTOP 20 SENSOR IDS")
display(df_smoke["id"].value_counts(dropna=False).head(20).to_frame("rows"))

print("\nROWS PER CITY")
city_coverage = (
    df_smoke.groupby("city", dropna=False)
    .agg(
        rows=("id", "size"),
        distinct_sensors=("id", "nunique"),
        distinct_time_values=("hr", "nunique"),
    )
    .sort_values("rows", ascending=False)
)
display(city_coverage)


UNIQUE COUNTS
city: 27 distinct non-missing values
  Missing: 0
id: 27 distinct non-missing values
  Missing: 0
hr: 24,485 distinct non-missing values
  Missing: 0

TOP 20 CITIES


,rows
city,
Cedar Rapids,234084
Atascadero,202320
Marysville,126180
Sarasota,109740
Denver,105936
Appleton,104664
Topeka,99372
Edmond,98088
Decatur,82416



TOP 20 SENSOR IDS


,rows
id,
PALinn2,234084
Atascadero,202320
PAMarysville 7th,126180
PASCG Air & Water Quality_EPA Air Sensor,109740
LTPP_CO,105936
PAAppleton,104664
PAKNI_E2,99372
LTPP_OK,98088
LTPP_GA,82416



ROWS PER CITY


,rows,distinct_sensors,distinct_time_values
city,,,
Cedar Rapids,234084,1,19507
Atascadero,202320,1,16860
Marysville,126180,1,10515
Sarasota,109740,1,9145
Denver,105936,1,8828
Appleton,104664,1,8722
Topeka,99372,1,8281
Edmond,98088,1,8174
Decatur,82416,1,6868


In [27]:

# Convert the source time field to a datetime for inspection.
# The original 'hr' column is preserved.
df_smoke["hr_parsed"] = pd.to_datetime(
    df_smoke["hr"],
    errors="coerce",
)

parse_failures = df_smoke["hr_parsed"].isna().sum()
valid_timestamps = df_smoke["hr_parsed"].dropna()

print("Original hr dtype:", df_smoke["hr"].dtype)
print("Timestamp parse failures:", f"{parse_failures:,}")
print("Timestamp parse failure percent:", f"{parse_failures / len(df_smoke) * 100:.4f}%")

if not valid_timestamps.empty:
    print("Earliest parsed timestamp:", valid_timestamps.min())
    print("Latest parsed timestamp:", valid_timestamps.max())
    print("Distinct parsed timestamps:", f"{valid_timestamps.nunique():,}")

    print("\nSample original and parsed values:")
    display(
        df_smoke[["hr", "hr_parsed"]]
        .drop_duplicates()
        .head(15)
    )

    print("\nRows by year and month:")
    time_coverage = (
        df_smoke.loc[df_smoke["hr_parsed"].notna()]
        .assign(
            year=df_smoke.loc[
                df_smoke["hr_parsed"].notna(), "hr_parsed"
            ].dt.year,
            month=df_smoke.loc[
                df_smoke["hr_parsed"].notna(), "hr_parsed"
            ].dt.month,
        )
        .groupby(["year", "month"])
        .size()
        .rename("rows")
        .reset_index()
    )
    display(time_coverage)


Original hr dtype: str
Timestamp parse failures: 0
Timestamp parse failure percent: 0.0000%
Earliest parsed timestamp: 2018-01-01 00:00:00
Latest parsed timestamp: 2020-10-20 11:00:00
Distinct parsed timestamps: 24,485

Sample original and parsed values:


,hr,hr_parsed
0,2018-12-31 00:00:00,2018-12-31 00:00:00
3,2018-12-31 01:00:00,2018-12-31 01:00:00
6,2018-12-31 02:00:00,2018-12-31 02:00:00
9,2018-12-31 03:00:00,2018-12-31 03:00:00
12,2018-12-31 04:00:00,2018-12-31 04:00:00
15,2018-12-31 05:00:00,2018-12-31 05:00:00
18,2018-12-31 06:00:00,2018-12-31 06:00:00
21,2018-12-31 07:00:00,2018-12-31 07:00:00
24,2018-12-31 08:00:00,2018-12-31 08:00:00
27,2018-12-31 09:00:00,2018-12-31 09:00:00



Rows by year and month:


,year,month,rows
0,2018,1,8700
1,2018,2,7872
2,2018,3,8544
3,2018,4,8556
4,2018,5,16500
5,2018,6,16488
6,2018,7,17436
7,2018,8,21000
8,2018,9,17232
9,2018,10,19956


In [28]:

valid_time_df = df_smoke.loc[df_smoke["hr_parsed"].notna()]

print("ROWS PER CITY: TIMESTAMP COVERAGE")
city_time_summary = (
    valid_time_df.groupby("city", dropna=False)
    .agg(
        rows=("hr_parsed", "size"),
        first_timestamp=("hr_parsed", "min"),
        last_timestamp=("hr_parsed", "max"),
        distinct_timestamps=("hr_parsed", "nunique"),
        distinct_sensors=("id", "nunique"),
    )
    .sort_values("rows", ascending=False)
)

display(city_time_summary)

# Examine the spacing between distinct timestamps in the dataset.
unique_times = (
    valid_time_df["hr_parsed"]
    .drop_duplicates()
    .sort_values()
)

if len(unique_times) > 1:
    time_differences = unique_times.diff().dropna()

    print("\nTIME DIFFERENCE DISTRIBUTION BETWEEN DISTINCT TIMESTAMPS")
    display(
        time_differences
        .value_counts()
        .head(15)
        .rename_axis("time_difference")
        .to_frame("occurrences")
    )


ROWS PER CITY: TIMESTAMP COVERAGE


,rows,first_timestamp,last_timestamp,distinct_timestamps,distinct_sensors
city,,,,,
Cedar Rapids,234084,2018-05-05 00:00:00,2020-10-08 11:00:00,19507,1
Atascadero,202320,2018-01-01 00:00:00,2020-10-19 09:00:00,16860,1
Marysville,126180,2018-10-25 00:00:00,2020-08-31 21:00:00,10515,1
Sarasota,109740,2019-05-30 03:00:00,2020-06-30 23:00:00,9145,1
Denver,105936,2019-08-14 08:00:00,2020-09-30 23:00:00,8828,1
Appleton,104664,2019-01-01 00:00:00,2020-01-09 16:00:00,8722,1
Topeka,99372,2019-03-12 14:00:00,2020-06-30 23:00:00,8281,1
Edmond,98088,2019-08-02 05:00:00,2020-09-30 23:00:00,8174,1
Decatur,82416,2019-08-01 12:00:00,2020-08-31 23:00:00,6868,1



TIME DIFFERENCE DISTRIBUTION BETWEEN DISTINCT TIMESTAMPS


,occurrences
time_difference,
0 days 01:00:00,24420
0 days 02:00:00,52
0 days 03:00:00,9
0 days 04:00:00,3


In [29]:

# Exact duplicates across all original columns, excluding our derived parsed time.
original_columns = [column for column in df_smoke.columns if column != "hr_parsed"]

exact_duplicate_count = int(
    df_smoke.duplicated(subset=original_columns, keep=False).sum()
)
exact_duplicate_extra_rows = int(
    df_smoke.duplicated(subset=original_columns, keep="first").sum()
)

print("Rows belonging to an exact-duplicate group:", f"{exact_duplicate_count:,}")
print("Extra rows beyond one copy of each exact record:", f"{exact_duplicate_extra_rows:,}")

# Candidate key: city + sensor ID + parsed timestamp.
candidate_key = ["city", "id", "hr_parsed"]

complete_key_rows = df_smoke.dropna(subset=candidate_key)

key_group_sizes = (
    complete_key_rows.groupby(candidate_key, dropna=False)
    .size()
    .rename("rows_in_group")
)

repeated_key_groups = key_group_sizes[key_group_sizes > 1]

print("\nCandidate key:", candidate_key)
print("Rows with all candidate key fields present:", f"{len(complete_key_rows):,}")
print("Distinct complete candidate keys:", f"{len(key_group_sizes):,}")
print("Repeated candidate-key groups:", f"{len(repeated_key_groups):,}")
print("Rows belonging to repeated candidate-key groups:", f"{int(repeated_key_groups.sum()):,}")
print("Extra rows beyond one per repeated key:", f"{int((repeated_key_groups - 1).sum()):,}")

print("\nLARGEST REPEATED KEY GROUPS")
display(repeated_key_groups.sort_values(ascending=False).head(20).to_frame())

if not repeated_key_groups.empty:
    example_keys = repeated_key_groups.sort_values(ascending=False).head(5).reset_index()
    example_keys = example_keys[candidate_key]

    print("\nEXAMPLE ROWS FROM REPEATED KEYS")
    example_rows = complete_key_rows.merge(
        example_keys,
        on=candidate_key,
        how="inner",
    )

    display(
        example_rows[
            [column for column in [
                "hr", "hr_parsed", "city", "id", "RH", "PA", "ref", "adj"
            ] if column in example_rows.columns]
        ].sort_values(candidate_key).head(50)
    )


Rows belonging to an exact-duplicate group: 0
Extra rows beyond one copy of each exact record: 0

Candidate key: ['city', 'id', 'hr_parsed']
Rows with all candidate key fields present: 1,513,776
Distinct complete candidate keys: 126,148
Repeated candidate-key groups: 126,148
Rows belonging to repeated candidate-key groups: 1,513,776
Extra rows beyond one per repeated key: 1,387,628

LARGEST REPEATED KEY GROUPS


rows_in_group
city       id               hr_parsed                         
Appleton   PAAppleton       2019-01-01 00:00:00             12
Marysville PAMarysville 7th 2019-07-06 16:00:00             12
                            2019-07-06 14:00:00             12
                            2019-07-06 13:00:00             12
                            2019-07-06 12:00:00             12
                            2019-07-06 11:00:00             12
                            2019-07-06 10:00:00             12
                            2019-07-06 09:00:00             12
                            2019-07-06 08:00:00             12
                            2019-07-06 07:00:00             12
                            2019-07-06 06:00:00             12
                            2019-07-06 05:00:00             12
                            2019-07-06 04:00:00             12
                            2019-07-06 03:00:00             12
                            2019-07-06 02:00:00             12
                            2019-07-06 01:00:00             12
                            2019-07-06 00:00:00             12
                            2019-07-05 23:00:00             12
                            2019-07-05 22:00:00             12
                            2019-07-05 21:00:00             12


EXAMPLE ROWS FROM REPEATED KEYS


,hr,hr_parsed,city,id,RH,PA,ref,adj
4,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,4.126561
9,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,3.390831
14,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,2.849754
19,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,3.778427
24,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,0.964891
29,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,0.553162
34,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,4.179951
39,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,3.443979
44,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,2.881255
49,2019-01-01 00:00:00,2019-01-01 00:00:00,Appleton,PAAppleton,60.466668,0.815111,2.0,3.823711


In [30]:

measurement_columns = ["PA", "ref", "adj", "RH"]

# Report how many values can be converted to numeric.
numeric_data = pd.DataFrame(index=df_smoke.index)

for column in measurement_columns:
    numeric_data[column] = pd.to_numeric(df_smoke[column], errors="coerce")

    source_nonmissing = df_smoke[column].notna().sum()
    numeric_nonmissing = numeric_data[column].notna().sum()
    conversion_failures = source_nonmissing - numeric_nonmissing

    print(
        f"{column}: source non-missing={source_nonmissing:,}; "
        f"numeric-conversion failures={conversion_failures:,}"
    )

print("\nNUMERIC SUMMARY")
display(numeric_data.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

print("\nNON-POSITIVE VALUES")
nonpositive_summary = pd.DataFrame({
    "non_positive_count": (numeric_data[["PA", "ref", "adj"]] <= 0).sum(),
    "non_positive_percent_of_nonmissing": (
        (numeric_data[["PA", "ref", "adj"]] <= 0).sum()
        / numeric_data[["PA", "ref", "adj"]].notna().sum()
        * 100
    ).round(4),
})
display(nonpositive_summary)

print("\nNEGATIVE RELATIVE HUMIDITY")
negative_rh = numeric_data["RH"] < 0
print("Count:", f"{negative_rh.sum():,}")

print("\nRELATIVE HUMIDITY ABOVE 100%")
rh_above_100 = numeric_data["RH"] > 100
print("Count:", f"{rh_above_100.sum():,}")

print("\nROWS WITH NON-POSITIVE PA OR REFERENCE VALUES")
flag_nonpositive = (
    (numeric_data["PA"] <= 0)
    | (numeric_data["ref"] <= 0)
)

display(
    df_smoke.loc[
        flag_nonpositive,
        [column for column in ["hr", "city", "id", "RH", "PA", "ref", "adj"] if column in df_smoke.columns]
    ].head(30)
)

print("\nROWS WITH RH OUTSIDE 0–100%")
flag_invalid_rh = (numeric_data["RH"] < 0) | (numeric_data["RH"] > 100)

display(
    df_smoke.loc[
        flag_invalid_rh,
        [column for column in ["hr", "city", "id", "RH", "PA", "ref", "adj"] if column in df_smoke.columns]
    ].head(30)
)


PA: source non-missing=1,513,776; numeric-conversion failures=0
ref: source non-missing=1,513,776; numeric-conversion failures=0
adj: source non-missing=1,513,776; numeric-conversion failures=0
RH: source non-missing=1,513,776; numeric-conversion failures=0

NUMERIC SUMMARY


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
PA,1513776.0,20.915392,67.154440,0.000000,0.106222,0.649333,3.357161,7.670667,16.572587,55.784667,342.197000,1561.375667
ref,1513776.0,13.724673,47.522034,-9.800000,-2.000000,1.000000,4.200000,6.900000,11.000000,29.000000,180.000000,1506.000000
adj,1513776.0,13.903669,46.886203,-1.984559,0.341484,1.959558,4.547440,6.356752,10.225906,29.839533,198.315754,2350.735775
RH,1513776.0,48.583639,18.350853,0.000000,8.103448,15.466667,35.066667,50.400000,63.783386,74.966668,79.100000,100.000000



NON-POSITIVE VALUES


,non_positive_count,non_positive_percent_of_nonmissing
PA,696,0.0460
ref,58992,3.8970
adj,3790,0.2504



NEGATIVE RELATIVE HUMIDITY
Count: 0

RELATIVE HUMIDITY ABOVE 100%
Count: 0

ROWS WITH NON-POSITIVE PA OR REFERENCE VALUES


,hr,city,id,RH,PA,ref,adj
506,2019-08-19 00:00:00,Atascadero,Atascadero,70.500000,2.694500,0.0,4.782430
516,2019-08-19 01:00:00,Atascadero,Atascadero,69.733330,3.955833,0.0,5.238172
526,2019-08-19 02:00:00,Atascadero,Atascadero,68.300000,2.992000,0.0,4.889767
536,2019-08-19 03:00:00,Atascadero,Atascadero,67.566670,3.902500,0.0,5.218867
546,2019-08-19 04:00:00,Atascadero,Atascadero,67.433330,5.132667,-1.0,5.664927
586,2019-08-19 08:00:00,Atascadero,Atascadero,59.000000,7.862333,0.0,6.660528
603,2019-08-19 10:00:00,Atascadero,Atascadero,38.733330,3.147000,-2.0,4.945729
606,2019-08-19 10:00:00,Marysville,PAMarysville 7th,39.800000,5.316500,0.0,5.731726
612,2019-08-19 11:00:00,Atascadero,Atascadero,34.666670,2.695167,-5.0,4.782670
671,2019-08-19 18:00:00,Atascadero,Atascadero,54.766670,2.146167,-1.0,4.584842



ROWS WITH RH OUTSIDE 0–100%


,hr,city,id,RH,PA,ref,adj


In [31]:

analysis_df = df_smoke.copy()

for column in ["PA", "ref", "adj", "RH"]:
    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="coerce",
    )

city_measurement_summary = (
    analysis_df.groupby("city", dropna=False)
    .agg(
        rows=("id", "size"),
        sensors=("id", "nunique"),
        PA_missing_pct=("PA", lambda x: round(x.isna().mean() * 100, 3)),
        ref_missing_pct=("ref", lambda x: round(x.isna().mean() * 100, 3)),
        RH_missing_pct=("RH", lambda x: round(x.isna().mean() * 100, 3)),
        PA_median=("PA", "median"),
        PA_99th_percentile=("PA", lambda x: x.quantile(0.99)),
        PA_max=("PA", "max"),
        ref_median=("ref", "median"),
        ref_99th_percentile=("ref", lambda x: x.quantile(0.99)),
        ref_max=("ref", "max"),
    )
    .sort_values("rows", ascending=False)
)

display(city_measurement_summary)


,rows,sensors,PA_missing_pct,ref_missing_pct,RH_missing_pct,PA_median,PA_99th_percentile,PA_max,ref_median,ref_99th_percentile,ref_max
city,,,,,,,,,,,
Cedar Rapids,234084,1,0.0,0.0,0.0,9.964668,53.105113,206.101000,6.500000,28.000000,162.000000
Atascadero,202320,1,0.0,0.0,0.0,4.732417,89.871833,685.585333,5.000000,56.000000,448.000000
Marysville,126180,1,0.0,0.0,0.0,6.040000,97.998112,233.395342,5.600000,45.000000,200.000000
Sarasota,109740,1,0.0,0.0,0.0,5.798621,38.279833,109.462800,6.400000,21.000000,34.900000
Denver,105936,1,0.0,0.0,0.0,5.018583,43.469500,70.014000,6.200000,28.300000,34.900000
Appleton,104664,1,0.0,0.0,0.0,8.249218,60.204310,80.456889,6.200000,28.300000,34.900000
Topeka,99372,1,0.0,0.0,0.0,7.982667,38.604167,73.597167,7.400000,25.500000,34.800000
Edmond,98088,1,0.0,0.0,0.0,8.142500,37.352667,66.758167,8.600000,27.000000,34.900000
Decatur,82416,1,0.0,0.0,0.0,11.897250,42.559833,63.011667,8.000000,21.500000,32.300000


In [32]:

group_columns = ["city", "id"]
variables_to_check = ["hr", "RH", "PA", "ref", "adj"]

for group_column in group_columns:
    print(f"\nMISSINGNESS BY {group_column.upper()}")

    missing_by_group = (
        df_smoke.groupby(group_column, dropna=False)[variables_to_check]
        .agg(lambda series: round(series.isna().mean() * 100, 3))
    )

    # Include group sizes so rates from tiny groups are not misinterpreted.
    group_sizes = (
        df_smoke.groupby(group_column, dropna=False)
        .size()
        .rename("row_count")
    )

    missing_by_group = (
        group_sizes.to_frame()
        .join(missing_by_group)
        .sort_values("row_count", ascending=False)
    )

    display(missing_by_group.head(30))

    print(f"\nGroups with the most missingness by {group_column}")
    display(
        missing_by_group
        .assign(
            mean_missing_pct=missing_by_group[variables_to_check].mean(axis=1)
        )
        .sort_values("mean_missing_pct", ascending=False)
        .head(15)
    )



MISSINGNESS BY CITY


,row_count,hr,RH,PA,ref,adj
city,,,,,,
Cedar Rapids,234084,0.0,0.0,0.0,0.0,0.0
Atascadero,202320,0.0,0.0,0.0,0.0,0.0
Marysville,126180,0.0,0.0,0.0,0.0,0.0
Sarasota,109740,0.0,0.0,0.0,0.0,0.0
Denver,105936,0.0,0.0,0.0,0.0,0.0
Appleton,104664,0.0,0.0,0.0,0.0,0.0
Topeka,99372,0.0,0.0,0.0,0.0,0.0
Edmond,98088,0.0,0.0,0.0,0.0,0.0
Decatur,82416,0.0,0.0,0.0,0.0,0.0



Groups with the most missingness by city


,row_count,hr,RH,PA,ref,adj,mean_missing_pct
city,,,,,,,
Cedar Rapids,234084,0.0,0.0,0.0,0.0,0.0,0.0
Bend_S,21096,0.0,0.0,0.0,0.0,0.0,0.0
Pinehurst_S,1884,0.0,0.0,0.0,0.0,0.0,0.0
Happy Camp_S,4176,0.0,0.0,0.0,0.0,0.0,0.0
RTP_S,5148,0.0,0.0,0.0,0.0,0.0,0.0
Oroville_S,12192,0.0,0.0,0.0,0.0,0.0,0.0
Missoula_S,13824,0.0,0.0,0.0,0.0,0.0,0.0
Forks of S._S,14388,0.0,0.0,0.0,0.0,0.0,0.0
El Portal_S,15348,0.0,0.0,0.0,0.0,0.0,0.0



MISSINGNESS BY ID


,row_count,hr,RH,PA,ref,adj
id,,,,,,
PALinn2,234084,0.0,0.0,0.0,0.0,0.0
Atascadero,202320,0.0,0.0,0.0,0.0,0.0
PAMarysville 7th,126180,0.0,0.0,0.0,0.0,0.0
PASCG Air & Water Quality_EPA Air Sensor,109740,0.0,0.0,0.0,0.0,0.0
LTPP_CO,105936,0.0,0.0,0.0,0.0,0.0
PAAppleton,104664,0.0,0.0,0.0,0.0,0.0
PAKNI_E2,99372,0.0,0.0,0.0,0.0,0.0
LTPP_OK,98088,0.0,0.0,0.0,0.0,0.0
LTPP_GA,82416,0.0,0.0,0.0,0.0,0.0



Groups with the most missingness by id


,row_count,hr,RH,PA,ref,adj,mean_missing_pct
id,,,,,,,
PALinn2,234084,0.0,0.0,0.0,0.0,0.0,0.0
Bend,21096,0.0,0.0,0.0,0.0,0.0,0.0
Alder WF,1884,0.0,0.0,0.0,0.0,0.0,0.0
Natchez WF,4176,0.0,0.0,0.0,0.0,0.0,0.0
RTP Rx,5148,0.0,0.0,0.0,0.0,0.0,0.0
Oroville,12192,0.0,0.0,0.0,0.0,0.0,0.0
MT,13824,0.0,0.0,0.0,0.0,0.0,0.0
Forks of Salmon,14388,0.0,0.0,0.0,0.0,0.0,0.0
El Portal,15348,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
import numpy as np

comparison_df = df_smoke[["PA", "ref", "adj"]].copy()

for column in comparison_df.columns:
    comparison_df[column] = pd.to_numeric(
        comparison_df[column],
        errors="coerce",
    )

print("PAIRWISE NON-MISSING COUNTS")
display(comparison_df.notna().sum().to_frame("non_missing_count"))

print("\nCORRELATIONS — DESCRIPTIVE ONLY")
display(comparison_df.corr().round(4))

print("\nPAIRWISE DIFFERENCE SUMMARY")
difference_summary = pd.DataFrame(index=["PA - ref", "adj - ref"])

for label, left, right in [
    ("PA - ref", "PA", "ref"),
    ("adj - ref", "adj", "ref"),
]:
    differences = comparison_df[left] - comparison_df[right]
    differences = differences.dropna()

    difference_summary.loc[label, "n"] = len(differences)
    difference_summary.loc[label, "mean_difference"] = differences.mean()
    difference_summary.loc[label, "median_difference"] = differences.median()
    difference_summary.loc[label, "MAE"] = differences.abs().mean()
    difference_summary.loc[label, "RMSE"] = np.sqrt((differences ** 2).mean())

display(difference_summary.round(4))


PAIRWISE NON-MISSING COUNTS


,non_missing_count
PA,1513776
ref,1513776
adj,1513776



CORRELATIONS — DESCRIPTIVE ONLY


,PA,ref,adj
PA,1.0000,0.9607,0.9686
ref,0.9607,1.0000,0.9606
adj,0.9686,0.9606,1.0000



PAIRWISE DIFFERENCE SUMMARY


,n,mean_difference,median_difference,MAE,RMSE
PA - ref,1513776.0,7.1907,1.4799,9.0343,26.2232
adj - ref,1513776.0,0.1790,-0.0674,3.5398,13.2592


In [34]:

humidity_df = df_smoke[["RH", "PA", "ref"]].copy()

for column in humidity_df.columns:
    humidity_df[column] = pd.to_numeric(
        humidity_df[column],
        errors="coerce",
    )

humidity_df["raw_difference"] = humidity_df["PA"] - humidity_df["ref"]

humidity_df = humidity_df.dropna(
    subset=["RH", "PA", "ref", "raw_difference"]
)

# Use only observed RH values in the conventional 0–100% range for this
# exploratory grouping; keep the excluded count visible.
valid_rh_mask = humidity_df["RH"].between(0, 100)

print("Rows available for humidity comparison:", f"{len(humidity_df):,}")
print("Rows with RH outside 0–100%:", f"{(~valid_rh_mask).sum():,}")

humidity_valid = humidity_df.loc[valid_rh_mask].copy()

humidity_valid["RH_band"] = pd.cut(
    humidity_valid["RH"],
    bins=[-0.001, 20, 40, 60, 80, 100],
    labels=["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"],
    include_lowest=True,
)

humidity_summary = (
    humidity_valid.groupby("RH_band", observed=False)
    .agg(
        observations=("raw_difference", "size"),
        median_RH=("RH", "median"),
        mean_raw_difference=("raw_difference", "mean"),
        median_raw_difference=("raw_difference", "median"),
        MAE=("raw_difference", lambda x: x.abs().mean()),
    )
)

display(humidity_summary.round(4))


Rows available for humidity comparison: 1,513,776
Rows with RH outside 0–100%: 0


,observations,median_RH,mean_raw_difference,median_raw_difference,MAE
RH_band,,,,,
0–20%,125724,14.1000,8.4793,-1.1333,12.0324
20–40%,363768,31.5333,8.6343,0.1268,11.0688
40–60%,538488,50.4667,6.1531,1.3569,7.9859
60–80%,475908,68.1724,7.0099,3.3547,7.9804
80–100%,9888,81.4667,2.9108,1.4829,3.8829


In [35]:

sensor_coverage = (
    df_smoke.groupby(["city", "id"], dropna=False)
    .agg(
        rows=("id", "size"),
        first_time=("hr_parsed", "min"),
        last_time=("hr_parsed", "max"),
        distinct_times=("hr_parsed", "nunique"),
        PA_missing_pct=("PA", lambda x: round(x.isna().mean() * 100, 3)),
        ref_missing_pct=("ref", lambda x: round(x.isna().mean() * 100, 3)),
    )
    .sort_values("rows", ascending=False)
)

print("NUMBER OF CITY-SENSOR GROUPS:", f"{len(sensor_coverage):,}")
print("\nTOP 30 CITY-SENSOR GROUPS BY ROW COUNT")
display(sensor_coverage.head(30))

print("\nDISTRIBUTION OF ROW COUNTS PER CITY-SENSOR GROUP")
display(
    sensor_coverage["rows"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("rows_per_city_sensor_group")
)


NUMBER OF CITY-SENSOR GROUPS: 27

TOP 30 CITY-SENSOR GROUPS BY ROW COUNT


,,rows,first_time,last_time,distinct_times,PA_missing_pct,ref_missing_pct
city,id,,,,,,
Cedar Rapids,PALinn2,234084,2018-05-05 00:00:00,2020-10-08 11:00:00,19507,0.0,0.0
Atascadero,Atascadero,202320,2018-01-01 00:00:00,2020-10-19 09:00:00,16860,0.0,0.0
Marysville,PAMarysville 7th,126180,2018-10-25 00:00:00,2020-08-31 21:00:00,10515,0.0,0.0
Sarasota,PASCG Air & Water Quality_EPA Air Sensor,109740,2019-05-30 03:00:00,2020-06-30 23:00:00,9145,0.0,0.0
Denver,LTPP_CO,105936,2019-08-14 08:00:00,2020-09-30 23:00:00,8828,0.0,0.0
Appleton,PAAppleton,104664,2019-01-01 00:00:00,2020-01-09 16:00:00,8722,0.0,0.0
Topeka,PAKNI_E2,99372,2019-03-12 14:00:00,2020-06-30 23:00:00,8281,0.0,0.0
Edmond,LTPP_OK,98088,2019-08-02 05:00:00,2020-09-30 23:00:00,8174,0.0,0.0
Decatur,LTPP_GA,82416,2019-08-01 12:00:00,2020-08-31 23:00:00,6868,0.0,0.0



DISTRIBUTION OF ROW COUNTS PER CITY-SENSOR GROUP


,rows_per_city_sensor_group
count,27.000000
mean,56065.777778
std,61517.961148
min,576.000000
10%,4759.200000
25%,14868.000000
50%,22380.000000
75%,98730.000000
90%,116316.000000
95%,179478.000000


In [36]:

relationship_df = df_smoke[["PA", "ref", "adj"]].copy()

for column in relationship_df.columns:
    relationship_df[column] = pd.to_numeric(
        relationship_df[column],
        errors="coerce",
    )

valid_adj_pa = relationship_df[["adj", "PA"]].dropna()
valid_adj_ref = relationship_df[["adj", "ref"]].dropna()

print("ROWS WHERE adj AND PA ARE BOTH AVAILABLE:", f"{len(valid_adj_pa):,}")
print(
    "Rows where adj equals PA exactly:",
    f"{(valid_adj_pa['adj'] == valid_adj_pa['PA']).sum():,}"
)
print(
    "Rows where adj differs from PA:",
    f"{(valid_adj_pa['adj'] != valid_adj_pa['PA']).sum():,}"
)

print("\nSAMPLE ROWS WHERE adj DIFFERS FROM PA")
different_adj_pa = (
    relationship_df["adj"].notna()
    & relationship_df["PA"].notna()
    & (relationship_df["adj"] != relationship_df["PA"])
)

display(
    df_smoke.loc[
        different_adj_pa,
        ["hr", "city", "id", "RH", "PA", "ref", "adj"]
    ].head(20)
)

print("\nCORRELATION OF ADJUSTED VALUES WITH REFERENCE")
if len(valid_adj_ref) > 1:
    print(valid_adj_ref["adj"].corr(valid_adj_ref["ref"]))

print(
    "\nNote: this is descriptive evidence only. Use the README or publication "
    "to establish how adj was produced before interpreting it."
)


ROWS WHERE adj AND PA ARE BOTH AVAILABLE: 1,513,776
Rows where adj equals PA exactly: 116
Rows where adj differs from PA: 1,513,660

SAMPLE ROWS WHERE adj DIFFERS FROM PA


,hr,city,id,RH,PA,ref,adj
0,2018-12-31 00:00:00,Atascadero,Atascadero,65.800000,43.397725,22.0,20.402327
1,2018-12-31 00:00:00,Cedar Rapids,PALinn2,54.644442,13.762445,7.0,8.861555
2,2018-12-31 00:00:00,Marysville,PAMarysville 7th,80.422221,50.088111,22.0,23.135010
3,2018-12-31 01:00:00,Atascadero,Atascadero,65.200000,30.376135,11.0,15.217768
4,2018-12-31 01:00:00,Cedar Rapids,PALinn2,52.711111,14.149000,6.0,9.006140
5,2018-12-31 01:00:00,Marysville,PAMarysville 7th,80.833332,23.814880,7.9,12.672775
6,2018-12-31 02:00:00,Atascadero,Atascadero,61.230770,27.511955,12.0,14.101239
7,2018-12-31 02:00:00,Cedar Rapids,PALinn2,50.666668,13.378666,9.0,8.718166
8,2018-12-31 02:00:00,Marysville,PAMarysville 7th,81.266668,29.492666,5.7,14.872451
9,2018-12-31 03:00:00,Atascadero,Atascadero,59.692310,28.043460,12.0,14.307782



CORRELATION OF ADJUSTED VALUES WITH REFERENCE
0.9606464686344419

Note: this is descriptive evidence only. Use the README or publication to establish how adj was produced before interpreting it.


In [37]:

# Keep only rows with successfully parsed timestamps.
monthly_df = df_smoke.loc[df_smoke["hr_parsed"].notna()].copy()

# Group observations by calendar month.
monthly_df["year_month"] = (
    monthly_df["hr_parsed"].dt.to_period("M").astype(str)
)

# Convert measurement fields to numeric for consistent calculations.
for column in ["PA", "ref", "RH", "adj"]:
    monthly_df[column] = pd.to_numeric(
        monthly_df[column],
        errors="coerce",
    )

# Summarize data coverage and missingness by month.
monthly_quality = (
    monthly_df.groupby("year_month")
    .agg(
        rows=("hr_parsed", "size"),
        distinct_sensors=("id", "nunique"),
        PA_missing_pct=("PA", lambda x: round(x.isna().mean() * 100, 3)),
        ref_missing_pct=("ref", lambda x: round(x.isna().mean() * 100, 3)),
        RH_missing_pct=("RH", lambda x: round(x.isna().mean() * 100, 3)),
        adj_missing_pct=("adj", lambda x: round(x.isna().mean() * 100, 3)),
        PA_median=("PA", "median"),
        ref_median=("ref", "median"),
    )
)

display(monthly_quality)


,rows,distinct_sensors,PA_missing_pct,ref_missing_pct,RH_missing_pct,adj_missing_pct,PA_median,ref_median
year_month,,,,,,,,
2018-01,8700,1,0.0,0.0,0.0,0.0,15.579890,6.000
2018-02,7872,1,0.0,0.0,0.0,0.0,8.789111,6.000
2018-03,8544,1,0.0,0.0,0.0,0.0,2.499667,1.000
2018-04,8556,1,0.0,0.0,0.0,0.0,3.528334,3.000
2018-05,16500,2,0.0,0.0,0.0,0.0,6.570889,4.500
2018-06,16488,2,0.0,0.0,0.0,0.0,5.738223,6.500
2018-07,17436,2,0.0,0.0,0.0,0.0,6.520556,7.000
2018-08,21000,3,0.0,0.0,0.0,0.0,22.113380,12.750
2018-09,17232,2,0.0,0.0,0.0,0.0,7.094754,7.000


In [38]:

## Investigate repeated records and correction/withholding types

# 1. Inspect how many rows belong to each adjustment and withholding type.
for column in ["adjtype", "wholdtype", "monitortype", "State", "week"]:
    print(f"\nVALUE COUNTS: {column}")
    print(
        df_smoke[column]
        .value_counts(dropna=False)
        .head(30)
        .to_string()
    )

# 2. Count distinct correction/withholding combinations.
print("\nCOUNTS BY adjtype AND wholdtype")
type_counts = (
    df_smoke.groupby(
        ["adjtype", "wholdtype"],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)
display(type_counts)

# 3. Determine how many rows exist per city-sensor-time combination.
# Do not remove or average these rows.
key_columns = ["city", "id", "hr"]

rows_per_observation = (
    df_smoke.groupby(key_columns, dropna=False)
    .size()
    .rename("rows_per_observation")
)

print("\nROWS PER CITY-SENSOR-TIME COMBINATION")
display(
    rows_per_observation
    .value_counts()
    .sort_index()
    .rename_axis("rows_per_observation")
    .to_frame("number_of_combinations")
)

# 4. Check whether each observation has multiple correction/withholding variants.
variant_counts = (
    df_smoke.groupby(key_columns, dropna=False)
    .agg(
        rows=("PA", "size"),
        distinct_adjtype=("adjtype", "nunique"),
        distinct_wholdtype=("wholdtype", "nunique"),
        distinct_adj_values=("adj", "nunique"),
        distinct_PA_values=("PA", "nunique"),
        distinct_ref_values=("ref", "nunique"),
    )
)

print("\nEXAMPLES WITH MULTIPLE ROWS PER OBSERVATION")
display(
    variant_counts.loc[variant_counts["rows"] > 1]
    .head(20)
)

# 5. Check whether the repeated rows agree on the original measurements.
# If PA, ref, and RH stay the same while adj or type fields vary,
# the rows likely represent different correction variants for the same reading.
print("\nWITHIN-OBSERVATION VARIATION")
display(
    variant_counts[
        [
            "rows",
            "distinct_adjtype",
            "distinct_wholdtype",
            "distinct_adj_values",
            "distinct_PA_values",
            "distinct_ref_values",
        ]
    ]
    .describe()
    .round(3)
)

# 6. Check State/week missingness against the withholding type.
print("\nSTATE AND WEEK MISSINGNESS BY WHOLDTYPE")
state_week_missingness = (
    df_smoke.groupby("wholdtype", dropna=False)
    .agg(
        rows=("PA", "size"),
        State_missing_pct=("State", lambda x: round(x.isna().mean() * 100, 3)),
        week_missing_pct=("week", lambda x: round(x.isna().mean() * 100, 3)),
        distinct_cities=("city", "nunique"),
        distinct_sensors=("id", "nunique"),
    )
    .sort_values("rows", ascending=False)
)
display(state_week_missingness)



VALUE COUNTS: adjtype
adjtype
C    252296
D    252296
E    252296
B    252296
A    252296
F    252296

VALUE COUNTS: wholdtype
wholdtype
LOBD    756888
LOSO    756888

VALUE COUNTS: monitortype
monitortype
Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation               545676
Teledyne T640 at 5.0 LPM - Broadband spectroscopy                     521904
not_specified                                                         240036
Teledyne T640X at 16.67 LPM - Broadband spectroscopy                  142944
Thermo Scientific TEOM 1405-DF Dichotomous FDMS - FDMS Gravimetric     43824
Thermo Scientific 1405-F FDMS w/VSCC - FDMS Gravimetric                19392

VALUE COUNTS: State
State
NaN                                         756888
PALinn2                                     117042
Atascadero                                  101160
PAMarysville 7th                             63090
PASCG Air & Water Quality_EPA Air Sensor     54870
LTPP_CO                                      5296

,adjtype,wholdtype,rows
0,A,LOBD,126148
1,A,LOSO,126148
2,B,LOBD,126148
3,B,LOSO,126148
4,C,LOBD,126148
5,C,LOSO,126148
6,D,LOBD,126148
7,D,LOSO,126148
8,E,LOBD,126148
9,E,LOSO,126148



ROWS PER CITY-SENSOR-TIME COMBINATION


,number_of_combinations
rows_per_observation,
12,126148



EXAMPLES WITH MULTIPLE ROWS PER OBSERVATION


rows  distinct_adjtype  \
city     id         hr                                            
Appleton PAAppleton 2019-01-01 00:00:00    12                 6   
                    2019-01-01 01:00:00    12                 6   
                    2019-01-01 02:00:00    12                 6   
                    2019-01-01 03:00:00    12                 6   
                    2019-01-01 04:00:00    12                 6   
                    2019-01-01 05:00:00    12                 6   
                    2019-01-01 06:00:00    12                 6   
                    2019-01-01 07:00:00    12                 6   
                    2019-01-01 08:00:00    12                 6   
                    2019-01-01 09:00:00    12                 6   
                    2019-01-01 10:00:00    12                 6   
                    2019-01-01 11:00:00    12                 6   
                    2019-01-01 12:00:00    12                 6   
                    2019-01-01 13:00:00    12                 6   
                    2019-01-01 14:00:00    12                 6   
                    2019-01-01 15:00:00    12                 6   
                    2019-01-01 16:00:00    12                 6   
                    2019-01-01 17:00:00    12                 6   
                    2019-01-01 18:00:00    12                 6   
                    2019-01-01 19:00:00    12                 6   

                                         distinct_wholdtype  \
city     id         hr                                        
Appleton PAAppleton 2019-01-01 00:00:00                   2   
                    2019-01-01 01:00:00                   2   
                    2019-01-01 02:00:00                   2   
                    2019-01-01 03:00:00                   2   
                    2019-01-01 04:00:00                   2   
                    2019-01-01 05:00:00                   2   
                    2019-01-01 06:00:00                   2   
                    2019-01-01 07:00:00                   2   
                    2019-01-01 08:00:00                   2   
                    2019-01-01 09:00:00                   2   
                    2019-01-01 10:00:00                   2   
                    2019-01-01 11:00:00                   2   
                    2019-01-01 12:00:00                   2   
                    2019-01-01 13:00:00                   2   
                    2019-01-01 14:00:00                   2   
                    2019-01-01 15:00:00                   2   
                    2019-01-01 16:00:00                   2   
                    2019-01-01 17:00:00                   2   
                    2019-01-01 18:00:00                   2   
                    2019-01-01 19:00:00                   2   

                                         distinct_adj_values  \
city     id         hr                                         
Appleton PAAppleton 2019-01-01 00:00:00                   11   
                    2019-01-01 01:00:00                   11   
                    2019-01-01 02:00:00                   11   
                    2019-01-01 03:00:00                   11   
                    2019-01-01 04:00:00                   11   
                    2019-01-01 05:00:00                   11   
                    2019-01-01 06:00:00                   11   
                    2019-01-01 07:00:00                   11   
                    2019-01-01 08:00:00                   11   
                    2019-01-01 09:00:00                   11   
                    2019-01-01 10:00:00                   11   
                    2019-01-01 11:00:00                   11   
                    2019-01-01 12:00:00                   11   
                    2019-01-01 13:00:00                   11   
                    2019-01-01 14:00:00                   11   
                    2019-01-01 15:00:00                   11   
                    2019-01


WITHIN-OBSERVATION VARIATION


,rows,distinct_adjtype,distinct_wholdtype,distinct_adj_values,distinct_PA_values,distinct_ref_values
count,126148.0,126148.0,126148.0,126148.000,126148.0,126148.0
mean,12.0,6.0,2.0,11.000,1.0,1.0
std,0.0,0.0,0.0,0.021,0.0,0.0
min,12.0,6.0,2.0,10.000,1.0,1.0
25%,12.0,6.0,2.0,11.000,1.0,1.0
50%,12.0,6.0,2.0,11.000,1.0,1.0
75%,12.0,6.0,2.0,11.000,1.0,1.0
max,12.0,6.0,2.0,11.000,1.0,1.0



STATE AND WEEK MISSINGNESS BY WHOLDTYPE


,rows,State_missing_pct,week_missing_pct,distinct_cities,distinct_sensors
wholdtype,,,,,
LOBD,756888,100.0,0.0,27,27
LOSO,756888,0.0,100.0,27,27


In [39]:

# Inspect one underlying hourly observation and all 12 variants.
example_key = (
    df_smoke[["city", "id", "hr"]]
    .drop_duplicates()
    .iloc[0]
)

mask = (
    df_smoke["city"].eq(example_key["city"])
    & df_smoke["id"].eq(example_key["id"])
    & df_smoke["hr"].eq(example_key["hr"])
)

variant_columns = [
    col for col in [
        "city", "id", "hr", "PA", "ref", "RH",
        "adj", "adjtype", "wholdtype", "monitortype",
        "State", "week"
    ]
    if col in df_smoke.columns
]

print("EXAMPLE HOURLY OBSERVATION")
display(df_smoke.loc[mask, variant_columns].sort_values(
    ["wholdtype", "adjtype"]
))

# Summarize the adjustment values by correction and withholding type.
print("\nADJUSTMENT SUMMARY BY TYPE")
adjustment_summary = (
    df_smoke.groupby(["wholdtype", "adjtype"], dropna=False)
    .agg(
        rows=("adj", "size"),
        distinct_sensors=("id", "nunique"),
        adj_mean=("adj", "mean"),
        adj_median=("adj", "median"),
        adj_min=("adj", "min"),
        adj_max=("adj", "max"),
        PA_mean=("PA", "mean"),
        ref_mean=("ref", "mean"),
    )
    .round(3)
    .reset_index()
)

display(adjustment_summary)

# Check whether each adjustment type has one consistent label/description
# in the dataset itself. The actual meaning may still require documentation.
print("\nADJUSTMENT TYPES")
display(
    df_smoke.groupby("adjtype", dropna=False)
    .agg(
        rows=("adj", "size"),
        unique_adjusted_values=("adj", "nunique"),
        mean_adjusted=("adj", "mean"),
        mean_raw_PA=("PA", "mean"),
        mean_reference=("ref", "mean"),
    )
    .round(3)
)

print("\nWITHHOLDING TYPES")
display(
    df_smoke.groupby("wholdtype", dropna=False)
    .agg(
        rows=("adj", "size"),
        missing_State_pct=("State", lambda x: x.isna().mean() * 100),
        missing_week_pct=("week", lambda x: x.isna().mean() * 100),
    )
    .round(3)
)


EXAMPLE HOURLY OBSERVATION


,city,id,hr,PA,ref,RH,adj,adjtype,wholdtype,monitortype,State,week
504592,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,22.818448,A,LOBD,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0
378444,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,21.255056,B,LOBD,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0
0,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,20.402327,C,LOBD,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0
126148,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,20.652527,D,LOBD,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0
252296,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,21.803489,E,LOBD,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0
630740,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,27.280667,F,LOBD,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,NaN,53.0
1350998,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,22.818448,A,LOSO,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Atascadero,NaN
1224850,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,21.356415,B,LOSO,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Atascadero,NaN
846406,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,20.368510,C,LOSO,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Atascadero,NaN
972554,Atascadero,Atascadero,2018-12-31 00:00:00,43.397725,22.0,65.8,20.912784,D,LOSO,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Atascadero,NaN



ADJUSTMENT SUMMARY BY TYPE


,wholdtype,adjtype,rows,distinct_sensors,adj_mean,adj_median,adj_min,adj_max,PA_mean,ref_mean
0,LOBD,A,126148,27,12.522,5.788,-1.985,820.098,20.915,13.725
1,LOBD,B,126148,27,13.737,6.479,3.249,1880.245,20.915,13.725
2,LOBD,C,126148,27,13.769,6.597,3.066,2350.736,20.915,13.725
3,LOBD,D,126148,27,13.739,6.482,2.051,1881.058,20.915,13.725
4,LOBD,E,126148,27,13.717,6.592,0.604,1852.420,20.915,13.725
5,LOBD,F,126148,27,15.942,5.560,0.000,1344.239,20.915,13.725
6,LOSO,A,126148,27,12.522,5.788,-1.985,820.098,20.915,13.725
7,LOSO,B,126148,27,13.714,6.506,3.276,1814.339,20.915,13.725
8,LOSO,C,126148,27,13.797,6.647,3.298,2298.118,20.915,13.725
9,LOSO,D,126148,27,13.707,6.485,2.104,1815.118,20.915,13.725



ADJUSTMENT TYPES


,rows,unique_adjusted_values,mean_adjusted,mean_raw_PA,mean_reference
adjtype,,,,,
A,252296,126106,12.522,20.915,13.725
B,252296,249880,13.725,20.915,13.725
C,252296,249890,13.783,20.915,13.725
D,252296,252289,13.723,20.915,13.725
E,252296,252289,13.717,20.915,13.725
F,252296,251861,15.952,20.915,13.725



WITHHOLDING TYPES


,rows,missing_State_pct,missing_week_pct
wholdtype,,,
LOBD,756888,100.0,0.0
LOSO,756888,0.0,100.0


In [40]:

preliminary_checks = [
    {
        "check": "Main dataset file",
        "result": "Present" if SMOKE_FILE.is_file() else "Missing",
        "interpretation": "Confirms the main CSV is available.",
    },
    {
        "check": "Missing values",
        "result": f"{int(df_smoke.isna().sum().sum()):,} missing cells",
        "interpretation": "Review per-column and grouped missingness.",
    },
    {
        "check": "Timestamp parse failures",
        "result": f"{int(df_smoke['hr_parsed'].isna().sum()):,}",
        "interpretation": "Inspect raw timestamp format if parsing failed.",
    },
    {
        "check": "Exact duplicate extra rows",
        "result": (
            f"{int(df_smoke.duplicated(subset=original_columns, keep='first').sum()):,}"
        ),
        "interpretation": "Review before deciding whether records are redundant.",
    },
    {
        "check": "Repeated city-sensor-time groups",
        "result": f"{len(repeated_key_groups):,} groups",
        "interpretation": "Investigate whether these are legitimate or duplicated observations.",
    },
    {
        "check": "Non-positive PA or reference values",
        "result": f"{int(flag_nonpositive.sum()):,} rows",
        "interpretation": "Flag for review; do not automatically delete.",
    },
    {
        "check": "RH outside 0–100%",
        "result": f"{int(flag_invalid_rh.sum()):,} rows",
        "interpretation": "Flag potentially invalid humidity values.",
    },
]

preliminary_checks_df = pd.DataFrame(preliminary_checks)
display(preliminary_checks_df)


,check,result,interpretation
0,Main dataset file,Present,Confirms the main CSV is available.
1,Missing values,"1,513,776 missing cells",Review per-column and grouped missingness.
2,Timestamp parse failures,0,Inspect raw timestamp format if parsing failed.
3,Exact duplicate extra rows,0,Review before deciding whether records are red...
4,Repeated city-sensor-time groups,"126,148 groups",Investigate whether these are legitimate or du...
5,Non-positive PA or reference values,"59,556 rows",Flag for review; do not automatically delete.
6,RH outside 0–100%,0 rows,Flag potentially invalid humidity values.


In [63]:
# Verify all 12 expected configurations per underlying observation

key = ["city", "id", "hr"]

expected_adjtypes = set(["A", "B", "C", "D", "E", "F"])
expected_wholdtypes = set(["LOBD", "LOSO"])

configuration_check = (
    df_smoke.groupby(key)
    .apply(
        lambda g: (
            len(g) == 12
            and set(g["adjtype"]) == expected_adjtypes
            and set(g["wholdtype"]) == expected_wholdtypes
            and not g.duplicated(["adjtype", "wholdtype"]).any()
        ),
        include_groups=False
    )
)

print("Total underlying observations:", len(configuration_check))
print("Observations with all 12 unique configurations:",
      configuration_check.sum())
print("Observations with incomplete or duplicate configurations:",
      (~configuration_check).sum())

Total underlying observations: 126148
Observations with all 12 unique configurations: 126148
Observations with incomplete or duplicate configurations: 0


In [64]:
# Compare adjusted values across configurations

adj_summary = (
    df_smoke.groupby(["adjtype", "wholdtype"])
    .agg(
        Rows=("adj", "size"),
        Missing_adj=("adj", lambda x: x.isna().sum()),
        Unique_adj_values=("adj", "nunique"),
        Mean_adj=("adj", "mean"),
        Median_adj=("adj", "median"),
        Min_adj=("adj", "min"),
        Max_adj=("adj", "max")
    )
)

display(adj_summary.round(3))

Rows  Missing_adj  Unique_adj_values  Mean_adj  \
adjtype wholdtype                                                     
A       LOBD       126148            0             126106    12.522   
        LOSO       126148            0             126106    12.522   
B       LOBD       126148            0             125700    13.737   
        LOSO       126148            0             124180    13.714   
C       LOBD       126148            0             125703    13.769   
        LOSO       126148            0             124187    13.797   
D       LOBD       126148            0             126147    13.739   
        LOSO       126148            0             126142    13.707   
E       LOBD       126148            0             126147    13.717   
        LOSO       126148            0             126142    13.717   
F       LOBD       126148            0             126023    15.942   
        LOSO       126148            0             125839    15.963   

                   Median_adj  Min_adj   Max_adj  
adjtype wholdtype                                 
A       LOBD            5.788   -1.985   820.098  
        LOSO            5.788   -1.985   820.098  
B       LOBD            6.479    3.249  1880.245  
        LOSO            6.506    3.276  1814.339  
C       LOBD            6.597    3.066  2350.736  
        LOSO            6.647    3.298  2298.118  
D       LOBD            6.482    2.051  1881.058  
        LOSO            6.485    2.104  1815.118  
E       LOBD            6.592    0.604  1852.420  
        LOSO            6.609    0.722  1794.992  
F       LOBD            5.560    0.000  1344.239  
        LOSO            5.542    0.000  1346.596

In [65]:
# Compare LOBD and LOSO adjusted values for the same observation

paired_adj = (
    df_smoke.pivot(
        index=["city", "id", "hr", "adjtype"],
        columns="wholdtype",
        values="adj"
    )
    .reset_index()
)

paired_adj["difference"] = (
    paired_adj["LOBD"] - paired_adj["LOSO"]
)

comparison = (
    paired_adj.groupby("adjtype")
    .agg(
        Total_pairs=("difference", "size"),
        Identical_pairs=("difference", lambda x: (x == 0).sum()),
        Different_pairs=("difference", lambda x: (x != 0).sum()),
        Mean_difference=("difference", "mean"),
        Mean_absolute_difference=("difference", lambda x: x.abs().mean()),
        Maximum_absolute_difference=("difference", lambda x: x.abs().max())
    )
)

display(comparison.round(4))

,Total_pairs,Identical_pairs,Different_pairs,Mean_difference,Mean_absolute_difference,Maximum_absolute_difference
adjtype,,,,,,
A,126148,126148,0,0.0000,0.0000,0.0000
B,126148,0,126148,0.0231,0.2418,75.5268
C,126148,0,126148,-0.0275,0.2668,114.2889
D,126148,0,126148,0.0319,0.2528,75.7631
E,126148,0,126148,0.0001,0.2825,83.7626
F,126148,58,126090,-0.0214,0.1773,25.6093


In [66]:
# Inspect the 10 largest adjusted values

extreme_adj = (
    df_smoke.nlargest(10, "adj")
    [["city", "id", "hr", "adjtype", "wholdtype",
      "PA", "ref", "adj", "RH"]]
    .sort_values("adj", ascending=False)
)

display(extreme_adj)

,city,id,hr,adjtype,wholdtype,PA,ref,adj,RH
115374,Oroville_S,Oroville,2020-09-12 07:00:00,C,LOBD,1561.375667,1506.0,2350.735775,44.233333
757256,Oroville_S,Oroville,2020-09-12 07:00:00,C,LOSO,1561.375667,1506.0,2298.117635,44.233333
115385,Oroville_S,Oroville,2020-09-12 08:00:00,C,LOBD,1536.696167,1480.0,2266.808485,42.666667
115574,Oroville_S,Oroville,2020-09-13 02:00:00,C,LOBD,1534.137833,1439.0,2258.232580,30.466667
115363,Oroville_S,Oroville,2020-09-12 06:00:00,C,LOBD,1533.221000,1493.0,2255.164867,44.866667
757257,Oroville_S,Oroville,2020-09-12 08:00:00,C,LOSO,1536.696167,1480.0,2215.710398,42.666667
757273,Oroville_S,Oroville,2020-09-13 02:00:00,C,LOSO,1534.137833,1439.0,2207.291776,30.466667
757255,Oroville_S,Oroville,2020-09-12 06:00:00,C,LOSO,1533.221000,1493.0,2204.280415,44.866667
115563,Oroville_S,Oroville,2020-09-13 01:00:00,C,LOBD,1498.685000,1419.0,2141.761063,31.366667
115586,Oroville_S,Oroville,2020-09-13 03:00:00,C,LOBD,1486.152167,1358.0,2101.635034,29.433333


In [67]:
# Calculate adjusted prediction errors

df_smoke["adj_error"] = df_smoke["adj"] - df_smoke["ref"]
df_smoke["abs_adj_error"] = df_smoke["adj_error"].abs()

error_summary = (
    df_smoke.groupby(["adjtype", "wholdtype"])
    .agg(
        Mean_Bias=("adj_error", "mean"),
        MAE=("abs_adj_error", "mean"),
        RMSE=("adj_error", lambda x: (x.pow(2).mean()) ** 0.5)
    )
)

display(error_summary.round(3))

Mean_Bias    MAE    RMSE
adjtype wholdtype                          
A       LOBD          -1.203  3.604  16.874
        LOSO          -1.203  3.604  16.874
B       LOBD           0.012  2.990  10.570
        LOSO          -0.011  3.020  10.028
C       LOBD           0.044  3.097  12.933
        LOSO           0.072  3.140  12.412
D       LOBD           0.014  2.928  10.563
        LOSO          -0.018  2.966  10.021
E       LOBD          -0.008  2.995  10.353
        LOSO          -0.008  3.046   9.884
F       LOBD           2.217  5.515  17.103
        LOSO           2.238  5.573  17.241

In [68]:
# Examine reference concentration distribution

display(
    df_smoke["ref"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    ).to_frame()
)

print("Zero or negative reference values:",
      (df_smoke["ref"] <= 0).sum())

print("Reference values above 100:",
      (df_smoke["ref"] > 100).sum())

print("Reference values above 250:",
      (df_smoke["ref"] > 250).sum())

print("Reference values above 500:",
      (df_smoke["ref"] > 500).sum())

,ref
count,1.513776e+06
mean,1.372467e+01
std,4.752203e+01
min,-9.800000e+00
50%,6.900000e+00
75%,1.100000e+01
90%,1.830000e+01
95%,2.900000e+01
99%,1.800000e+02
99.9%,7.090000e+02


Zero or negative reference values: 58992
Reference values above 100: 24000
Reference values above 250: 10884
Reference values above 500: 3252


In [69]:
# Create one record per underlying hourly observation

base_smoke = df_smoke.drop_duplicates(
    subset=["city", "id", "hr"]
).copy()

# Count non-positive reference observations by city

negative_ref_city = (
    base_smoke.assign(
        Nonpositive_ref=base_smoke["ref"] <= 0
    )
    .groupby("city")
    .agg(
        Total_observations=("ref", "size"),
        Nonpositive_ref=("Nonpositive_ref", "sum")
    )
)

negative_ref_city["Nonpositive_pct"] = (
    negative_ref_city["Nonpositive_ref"] /
    negative_ref_city["Total_observations"] * 100
)

display(
    negative_ref_city.sort_values(
        "Nonpositive_ref",
        ascending=False
    ).round(2)
)

,Total_observations,Nonpositive_ref,Nonpositive_pct
city,,,
Atascadero,16860,3071,18.21
Cedar Rapids,19507,556,2.85
Marysville,10515,533,5.07
Tulelake_S,1720,241,14.01
Hoopa_S,1632,161,9.87
Missoula,1738,88,5.06
Forks of S._S,1199,55,4.59
El Portal_S,1279,53,4.14
Appleton,8722,49,0.56


In [70]:
# Investigate non-positive reference measurements over time
# in the five cities with the most non-positive observations

top_cities = [
    "Atascadero",
    "Cedar Rapids",
    "Marysville",
    "Tulelake_S",
    "Hoopa_S"
]

base_smoke["hr"] = pd.to_datetime(base_smoke["hr"])
base_smoke["YearMonth"] = base_smoke["hr"].dt.to_period("M")

city_month_check = (
    base_smoke[base_smoke["city"].isin(top_cities)]
    .assign(
        Nonpositive_ref=lambda x: x["ref"] <= 0
    )
    .groupby(["city", "YearMonth"])
    .agg(
        Total_observations=("ref", "size"),
        Nonpositive_ref=("Nonpositive_ref", "sum")
    )
)

city_month_check["Nonpositive_pct"] = (
    city_month_check["Nonpositive_ref"] /
    city_month_check["Total_observations"] * 100
)

display(
    city_month_check[
        city_month_check["Nonpositive_ref"] > 0
    ].round(2)
)

Total_observations  Nonpositive_ref  Nonpositive_pct
city       YearMonth                                                      
Atascadero 2018-01                   725              136            18.76
           2018-02                   656               84            12.80
           2018-03                   712              300            42.13
           2018-04                   713              173            24.26
           2018-05                   730              295            40.41
...                                  ...              ...              ...
Marysville 2020-08                   490               12             2.45
Tulelake_S 2020-07                     8                2            25.00
           2020-08                   681              119            17.47
           2020-09                   614               49             7.98
           2020-10                   417               71            17.03

[78 rows x 3 columns]

In [71]:
# Summarize monthly non-positive reference patterns by city

monthly_pattern = (
    city_month_check
    .groupby("city")
    .agg(
        Months_with_nonpositive=("Nonpositive_ref", lambda x: (x > 0).sum()),
        Total_nonpositive=("Nonpositive_ref", "sum"),
        Highest_monthly_pct=("Nonpositive_pct", "max"),
        Average_monthly_pct=("Nonpositive_pct", "mean")
    )
)

display(monthly_pattern.round(2))

,Months_with_nonpositive,Total_nonpositive,Highest_monthly_pct,Average_monthly_pct
city,,,,
Atascadero,25,3071,42.13,18.28
Cedar Rapids,28,556,15.90,2.78
Hoopa_S,4,161,28.57,14.16
Marysville,17,533,22.63,4.81
Tulelake_S,4,241,25.00,16.87


In [72]:
# Missingness by withholding approach

missing_by_withholding = (
    df_smoke.groupby("wholdtype")
    .apply(
        lambda g: g.isna().sum(),
        include_groups=False
    )
    .T
)

display(missing_by_withholding)

wholdtype,LOBD,LOSO
Unnamed: 0,0,0
hr,0,0
city,0,0
id,0,0
RH,0,0
PA,0,0
ref,0,0
monitortype,0,0
State,756888,0
week,0,756888


In [73]:
# Verify underlying measurements across 12 configurations

measurement_consistency = (
    df_smoke.groupby(["city", "id", "hr"])
    .agg(
        PA_unique=("PA", "nunique"),
        ref_unique=("ref", "nunique"),
        RH_unique=("RH", "nunique")
    )
)

print("Observations:", len(measurement_consistency))
print("PA inconsistent:",
      (measurement_consistency["PA_unique"] > 1).sum())
print("ref inconsistent:",
      (measurement_consistency["ref_unique"] > 1).sum())
print("RH inconsistent:",
      (measurement_consistency["RH_unique"] > 1).sum())

Observations: 126148
PA inconsistent: 0
ref inconsistent: 0
RH inconsistent: 0


# 6.6 Data Profile Findings — Hourly Wildfire-Smoke Dataset

### 1. Source and Purpose

- **Dataset:** `Finaldataset_correcteddatasetwholding_10_26_22.csv`
- **Source:** U.S. EPA dataset accompanying *Correction and Accuracy of PurpleAir PM2.5 Measurements for Extreme Wildfire Smoke*.
- **Downloaded:** October 22, 2026.
- **Local directory:** `data/raw/hourly_smoke/`
- **Purpose:** Evaluate how raw and corrected low-cost PurpleAir PM2.5 measurements agree with reference-monitor measurements, particularly under smoke-impacted conditions.

The accompanying README explains that the dataset contains measurements corrected using withholding procedures. A correction is developed while withholding a site or a week, then applied to the withheld site or period.

### 2. Dataset Size and Coverage

- **Rows:** 1,513,776
- **Original columns:** 13
- **Time coverage:** January 1, 2018 – October 20, 2020
- **Distinct city–sensor–timestamp combinations:** 126,148
- **Distinct city–sensor groups:** 27

The dataset covers multiple locations and time periods. Its rows do not represent independent hourly observations because each underlying observation appears under multiple correction and withholding configurations.

### 3. What One Row Represents

The underlying measurement is identified by `city`, `id`, and `hr`.

Each of the 126,148 underlying combinations appears exactly 12 times:

- Six adjustment types: A, B, C, D, E, and F.
- Two withholding types: LOBD and LOSO.

A configuration-completeness check confirmed that all 126,148 underlying observations contain exactly 12 unique configurations. There are no missing or duplicate configuration combinations.

A separate consistency check confirmed that `PA`, `ref`, and `RH` remain identical across all 12 configurations for every underlying observation.

The `adj` variable is configuration-dependent and can differ across adjustment types and withholding approaches.

Therefore, the 12 rows should not be treated as 12 independent sensor observations. They represent alternative correction and withholding configurations applied to the same underlying measurements.

**Decision:** Preserve all original configuration records. Do not average the 12 configurations or randomly split their variants across training and testing. Comparisons will be made using matched underlying observations and clearly defined configurations.

### 4. Key Variables and Documentation

| Column | Meaning |
|---|---|
| `hr` | Hourly timestamp |
| `city` | City where the sensor/reference-monitor pair is located |
| `id` | PurpleAir sensor identifier |
| `PA` | Raw PurpleAir CF=1 PM2.5 concentration, µg/m³ |
| `ref` | Reference-monitor PM2.5 concentration, µg/m³ |
| `RH` | Relative humidity, % |
| `adj` | Adjusted PM2.5 concentration produced using the supplied correction procedure |
| `adjtype` | Correction model used, A–F |
| `wholdtype` | Withholding/evaluation approach |
| `monitortype` | Reference-monitor type, where available |
| `State` | State identifier used for site-withholding records |
| `week` | Week of year used for leave-one-week-out withholding |

The six adjustment types are described in the supplied source documentation as follows:

- **A:** Existing U.S.-wide correction incorporating relative humidity.
- **B:** Quadratic correction.
- **C:** Cubic correction.
- **D:** Quadratic correction with relative humidity.
- **E:** Quadratic correction with relative humidity and a PM2.5 × RH interaction.
- **F:** Relative-humidity growth model based on the Nilson approach, with RH limited to 30–70%.

The two withholding approaches are:

- **LOBD:** Leave out by date, withholding a week of the year.
- **LOSO:** Leave out by site, withholding by city.

These are published correction configurations, not six independently trained models developed in this project. The `adj` column contains their supplied adjusted outputs.

### 5. Missingness

The overall missingness assessment identified missing values in two columns:

- `State`: 756,888 missing values (50% of all rows).
- `week`: 756,888 missing values (50% of all rows).

Grouped investigation confirmed:

- `State` is missing in all LOBD records and populated in all LOSO records.
- `week` is missing in all LOSO records and populated in all LOBD records.
- All other examined columns have zero missing values, including `PA`, `ref`, `RH`, and `adj`.

This pattern is consistent with the two withholding approaches. The missing values are structural rather than evidence of unexpected measurement-data loss.

**Decision:** Retain `State` and `week` without automatic imputation. Preserve their structural missingness and use the applicable identifier for the corresponding withholding approach.

### 6. Duplicates and Repeated Observations

- Exact duplicate rows across the original dataset: **0**.
- Underlying city–sensor–timestamp combinations: **126,148**.
- Configuration records per underlying observation: **12**.
- Incomplete or duplicate configuration combinations: **0**.

The repeated-record investigation established that:

- `PA`, `ref`, and `RH` are identical across all 12 configurations for every underlying observation.
- `adj` changes across correction configurations and, for most adjustment types, between LOBD and LOSO.
- The repeated rows represent alternative correction and withholding results, not independent hourly measurements.

A paired comparison of LOBD and LOSO adjusted values showed:

| Adjustment type | Identical pairs | Different pairs | Mean absolute difference |
|---|---:|---:|---:|
| A | 126,148 | 0 | 0.0000 |
| B | 0 | 126,148 | 0.2418 |
| C | 0 | 126,148 | 0.2668 |
| D | 0 | 126,148 | 0.2528 |
| E | 0 | 126,148 | 0.2825 |
| F | 58 | 126,090 | 0.1773 |

Adjustment A produced identical values under both withholding approaches. Types B–E differed for every paired observation, while type F differed for all but 58 observations.

**Decision:** Retain the original configuration records. Evaluate LOBD and LOSO separately because they represent different withholding strategies and are not interchangeable for adjustment types B–F. Type A is identical under both approaches in this dataset.

### 7. Measurement Ranges and Values Requiring Investigation

The reference measurement distribution is strongly right-skewed.

| Statistic | Reference PM2.5 (µg/m³) |
|---|---:|
| Count | 1,513,776 configuration rows |
| Mean | 13.72 |
| Standard deviation | 47.52 |
| Minimum | -9.80 |
| Median | 6.90 |
| 75th percentile | 11.00 |
| 90th percentile | 18.30 |
| 95th percentile | 29.00 |
| 99th percentile | 180.00 |
| 99.9th percentile | 709.00 |
| Maximum | 1,506.00 |

The mean is substantially higher than the median, indicating a right-skewed distribution. The sharp increase between the 95th and 99th percentiles indicates a relatively small number of extremely high-concentration observations.

Because each underlying observation appears 12 times, the following counts are expressed at the underlying observation level:

| Reference condition | Configuration rows | Underlying observations |
|---|---:|---:|
| Zero or negative | 58,992 | 4,916 |
| Above 100 µg/m³ | 24,000 | 2,000 |
| Above 250 µg/m³ | 10,884 | 907 |
| Above 500 µg/m³ | 3,252 | 271 |

#### Non-positive Reference Measurements

A total of 4,916 underlying observations have reference values less than or equal to zero. These observations are not evenly distributed across locations.

The largest concentrations were found in:

- Atascadero: 3,071 observations, approximately 62.5% of all non-positive observations.
- Cedar Rapids: 556 observations.
- Marysville: 533 observations.
- Tulelake_S: 241 observations.
- Hoopa_S: 161 observations.

Further monthly investigation showed different patterns across cities:

| City | Months with non-positive readings | Total non-positive observations | Highest monthly percentage |
|---|---:|---:|---:|
| Atascadero | 25 | 3,071 | 42.13% |
| Cedar Rapids | 28 | 556 | 15.90% |
| Hoopa_S | 4 | 161 | 28.57% |
| Marysville | 17 | 533 | 22.63% |
| Tulelake_S | 4 | 241 | 25.00% |

Atascadero has the largest number of affected observations and a persistent pattern across 25 months. Cedar Rapids also has affected readings across many months, although its average monthly percentage is relatively low. Hoopa_S and Tulelake_S show higher monthly percentages concentrated in four months each.

These findings indicate geographic and temporal concentration of non-positive reference measurements.

**Decision:** Retain and flag all original observations. Do not automatically replace or remove non-positive values. Their treatment will depend on the evaluation metric and documented analytical rules. Percentage-based error metrics will require particular care because zero and negative reference values make percentage errors undefined or misleading.

#### Extreme Adjusted Values

Inspection of the ten largest adjusted values showed that the highest values were concentrated in Oroville_S during September 12–13, 2020.

The highest inspected observation had:

- Raw PurpleAir (`PA`): 1,561.38 µg/m³.
- Reference (`ref`): 1,506.00 µg/m³.
- Adjusted (`adj`): 2,350.74 µg/m³.
- Adjustment type: C.
- Withholding approach: LOBD.

The underlying raw and reference measurements were both extremely high, consistent with a severe smoke episode. However, the adjusted prediction substantially exceeded the reference measurement.

**Decision:** Preserve the original measurements and adjusted values. Do not automatically classify extreme observations as errors, cap them, or remove them. Investigate their influence on model-performance metrics, especially RMSE and performance under high-concentration smoke conditions.

### 8. Preliminary Evaluation of Supplied Adjustments

The supplied adjusted values were compared with the reference measurements using:

- **Mean Bias:** Adjusted prediction minus reference.
- **MAE:** Mean absolute error.
- **RMSE:** Root mean squared error.

Results by adjustment and withholding approach:

| Adjustment | Withholding | Mean Bias | MAE | RMSE |
|---|---|---:|---:|---:|
| A | LOBD | -1.203 | 3.604 | 16.874 |
| A | LOSO | -1.203 | 3.604 | 16.874 |
| B | LOBD | 0.012 | 2.990 | 10.570 |
| B | LOSO | -0.011 | 3.020 | 10.028 |
| C | LOBD | 0.044 | 3.097 | 12.933 |
| C | LOSO | 0.072 | 3.140 | 12.412 |
| D | LOBD | 0.014 | 2.928 | 10.563 |
| D | LOSO | -0.018 | 2.966 | 10.021 |
| E | LOBD | -0.008 | 2.995 | 10.353 |
| E | LOSO | -0.008 | 3.046 | 9.884 |
| F | LOBD | 2.217 | 5.515 | 17.103 |
| F | LOSO | 2.238 | 5.573 | 17.241 |

The descriptive results show:

- Adjustment D achieved the lowest MAE under both LOBD (2.928 µg/m³) and LOSO (2.966 µg/m³).
- Adjustment E achieved the lowest RMSE under LOSO (9.884 µg/m³).
- Adjustment F had the highest MAE and RMSE under both withholding approaches and showed positive mean bias of approximately 2.2 µg/m³.
- Adjustment A showed negative mean bias of -1.203 µg/m³ under both approaches.
- Adjustment C had relatively high RMSE, consistent with the extreme adjusted predictions identified during the Oroville smoke episode.

These are descriptive comparisons of the supplied correction outputs against the reference measurements. They do not establish the performance of independently trained models developed in this project. No overall winning adjustment has been selected.

The MAE and RMSE results should be interpreted alongside the highly skewed reference distribution, particularly because a relatively small number of extreme observations can have a substantial influence on RMSE.

### 9. Data-Quality Decisions

| Issue | Decision | Reason |
|---|---|---|
| Repeated sensor-time configurations | Preserve in raw data; compare using matched observations | Twelve configurations represent alternative correction and withholding results |
| Missing `State` and `week` | Retain without imputation | Structural missingness associated with withholding type |
| Non-positive reference measurements | Retain and flag | Concentrated geographically and temporally; metric-specific handling required |
| Extreme PM2.5 readings | Retain and investigate | High concentrations are central to wildfire-smoke analysis |
| Extreme adjusted predictions | Retain and flag | Some corrections substantially exceed reference measurements |
| Supplied `adj` values | Use as published comparison results | They are not independently trained predictions from this project |
| LOBD and LOSO | Evaluate separately | Different withholding strategies produce different results for B–F |
| Configuration completeness | Verified | All 126,148 observations contain all 12 unique configurations |
| Underlying measurement consistency | Verified | `PA`, `ref`, and `RH` remain identical across configurations |

### 10. Limitations and Modeling Implications

1. The dataset contains multiple correction configurations for the same underlying observation. Comparisons must explicitly identify the adjustment type and withholding approach.
2. The source paper's correction results provide useful benchmarks, but they must not be presented as results from newly trained models in this project.
3. Rows sharing the same underlying city–sensor–hour observation must be kept together when creating analytical partitions. Configuration variants must never be treated as independent observations.
4. If evaluating generalization to unseen sites, a site-level holdout is appropriate. If evaluating generalization to unseen periods, a time-based holdout is appropriate. The selected approach must match the research question.
5. Reference monitors are the comparison target, but they should not be assumed to be perfectly error-free ground truth.
6. The dataset contains 27 city–sensor groups, so findings may not generalize to all PurpleAir sensors, locations, climates, or wildfire-smoke events.
7. The strongly right-skewed reference distribution and geographically concentrated non-positive readings may affect model performance and error metrics.
8. High-concentration smoke episodes may produce different calibration behavior from ordinary pollution conditions. Performance should therefore be examined across concentration ranges.

### 11. Question for the Data Owner

**Question:** For a new analysis using this dataset, which adjustment type and withholding configuration should be treated as the primary benchmark, and are there additional quality-control or site-selection rules that should be applied before comparing raw and corrected PM2.5 measurements?

**Documentation checked first:** The supplied README and the paper *Correction and Accuracy of PurpleAir PM2.5 Measurements for Extreme Wildfire Smoke*. The files describe the correction and withholding configurations, but the intended primary benchmark for this separate research question should be stated explicitly.

### 12. Current Status

Dataset B has been investigated for structure, time coverage, configuration completeness, measurement consistency, missingness, repeated observations, reference concentration distribution, non-positive measurements, extreme adjusted values, and supplied correction performance.

The structural and missingness investigations are complete. The original observations and all correction configurations are preserved.

The supplied correction results have been summarized, but no new calibration model has been independently trained or validated using this dataset. Final analytical decisions concerning reference-value handling, evaluation metrics, and grouped validation will be documented before model development.

In [74]:

# ================================================================
# FINAL VERIFICATION FOR THE DATA PROFILE CARD
# Dataset A: 24-Hour Collocation
# Dataset B: Hourly Wildfire-Smoke
#
# This cell summarizes and verifies existing data.
# It does NOT modify, delete, or impute original observations.
# ================================================================

import pandas as pd
import numpy as np

# ------------------------------------------------
# DATASET A: 24-HOUR COLLOCATION
# ------------------------------------------------

print("\n" + "=" * 75)
print("DATASET A — 24-HOUR COLLOCATION")
print("=" * 75)

key_a = ["ID", "Date", "region"]

print("\n1. DATASET STRUCTURE")
print("Rows:", f"{len(df_24hr):,}")
print("Columns:", len(df_24hr.columns))
print("Exact duplicate rows:", f"{df_24hr.duplicated().sum():,}")

print("Date range:",
      df_24hr["Date"].min(),
      "to",
      df_24hr["Date"].max())

a_groups = df_24hr.groupby(key_a, dropna=False)

print("\n2. OBSERVATION STRUCTURE")
print("Distinct ID-Date-region combinations:",
      f"{len(a_groups):,}")

print("Repeated ID-Date-region combinations:",
      f"{(a_groups.size() > 1).sum():,}")

print("Rows belonging to repeated groups:",
      f"{a_groups.size()[a_groups.size() > 1].sum():,}")

print("\n3. MISSINGNESS")

missing_a = pd.DataFrame({
    "Missing_count": df_24hr.isna().sum(),
    "Missing_percent": (
        df_24hr.isna().mean() * 100
    ).round(3)
}).sort_values("Missing_percent", ascending=False)

display(missing_a)

print("\n4. NON-POSITIVE REFERENCE VALUES")

pm25_a = pd.to_numeric(
    df_24hr["PM25FM"],
    errors="coerce"
)

bad_a = pm25_a <= 0

affected_a = (
    df_24hr.loc[bad_a, key_a]
    .drop_duplicates()
)

print("Non-positive PM25FM rows:",
      f"{bad_a.sum():,}")

print("Distinct ID-Date-region groups affected:",
      f"{len(affected_a):,}")

print("\n5. ENVIRONMENTAL MISSINGNESS")

environment_a = pd.DataFrame({
    "Missing_count": df_24hr[
        ["RH", "TempC", "Dewpoint"]
    ].isna().sum(),

    "Missing_percent": (
        df_24hr[
            ["RH", "TempC", "Dewpoint"]
        ].isna().mean() * 100
    ).round(3)
})

display(environment_a)

print("Rows missing both RH and TempC:",
      df_24hr[["RH", "TempC"]].isna().all(axis=1).sum())

print("Rows missing TempC only:",
      (df_24hr["TempC"].isna() &
       df_24hr["RH"].notna()).sum())

print("Rows missing RH only:",
      (df_24hr["RH"].isna() &
       df_24hr["TempC"].notna()).sum())


# ------------------------------------------------
# DATASET B: HOURLY WILDFIRE-SMOKE
# ------------------------------------------------

print("\n" + "=" * 75)
print("DATASET B — HOURLY WILDFIRE-SMOKE")
print("=" * 75)

key_b = ["city", "id", "hr"]

# Original file contains 13 columns.
# Working dataframe may contain additional derived columns.

print("\n1. DATASET STRUCTURE")

print("Rows:", f"{len(df_smoke):,}")
print("Original dataset columns: 13")
print("Current working dataframe columns:",
      len(df_smoke.columns))

print("Exact duplicate rows:",
      f"{df_smoke.duplicated().sum():,}")

print("Additional working columns:",
      [
          col for col in df_smoke.columns
          if col not in [
              "Unnamed: 0", "hr", "city", "id",
              "RH", "PA", "ref", "monitortype",
              "State", "week", "adj",
              "adjtype", "wholdtype"
          ]
      ])


# ------------------------------------------------
# 2. TIMESTAMP VERIFICATION
# ------------------------------------------------

print("\n2. TIMESTAMP VERIFICATION")

hr_original = pd.to_datetime(
    df_smoke["hr"],
    errors="coerce"
)

print("\nTimestamp column: hr")
print("Parse failures:", f"{hr_original.isna().sum():,}")
print("Minimum:", hr_original.min())
print("Maximum:", hr_original.max())

if "hr_parsed" in df_smoke.columns:

    hr_parsed = pd.to_datetime(
        df_smoke["hr_parsed"],
        errors="coerce"
    )

    print("\nTimestamp column: hr_parsed")
    print("Parse failures:", f"{hr_parsed.isna().sum():,}")
    print("Minimum:", hr_parsed.min())
    print("Maximum:", hr_parsed.max())

    timestamp_comparison = pd.DataFrame({
        "hr": hr_original,
        "hr_parsed": hr_parsed
    })

    timestamp_comparison["Same_timestamp"] = (
        timestamp_comparison["hr"] ==
        timestamp_comparison["hr_parsed"]
    )

    print("\nTimestamp agreement:")
    print(
        timestamp_comparison[
            "Same_timestamp"
        ].value_counts(dropna=False)
    )

    print(
        "\nRows with different timestamps:",
        (
            timestamp_comparison[
                "Same_timestamp"
            ] == False
        ).sum()
    )

    if (
        timestamp_comparison["Same_timestamp"].all()
    ):
        print(
            "\nRESULT: hr and hr_parsed agree for all rows."
        )
    else:
        print(
            "\nWARNING: Timestamp discrepancy detected. "
            "Inspect differing rows before finalizing "
            "the documented date range."
        )

        display(
            timestamp_comparison.loc[
                ~timestamp_comparison["Same_timestamp"]
            ].head(10)
        )

else:
    print(
        "\nhr_parsed is not available in the working dataframe."
    )


# ------------------------------------------------
# 3. UNDERLYING OBSERVATION STRUCTURE
# ------------------------------------------------

print("\n3. UNDERLYING OBSERVATION STRUCTURE")

b_groups = df_smoke.groupby(
    key_b,
    dropna=False
)

group_sizes = b_groups.size()

print("Distinct city-sensor-time combinations:",
      f"{len(b_groups):,}")

print("\nRows per underlying observation:")

display(
    group_sizes.value_counts()
    .sort_index()
    .rename_axis("Rows_per_observation")
    .to_frame("Number_of_observations")
)

print("Expected rows per observation: 12")

print("Observations with exactly 12 rows:",
      f"{(group_sizes == 12).sum():,}")

print("Observations with other row counts:",
      f"{(group_sizes != 12).sum():,}")


# ------------------------------------------------
# 4. CONFIGURATION COMPLETENESS
# ------------------------------------------------

print("\n4. CONFIGURATION COMPLETENESS")

expected_adjtypes = {
    "A", "B", "C", "D", "E", "F"
}

expected_wholdtypes = {
    "LOBD", "LOSO"
}

configuration_check = (
    df_smoke.groupby(key_b)
    .apply(
        lambda g: (
            len(g) == 12
            and set(g["adjtype"]) == expected_adjtypes
            and set(g["wholdtype"]) == expected_wholdtypes
            and not g.duplicated(
                ["adjtype", "wholdtype"]
            ).any()
        ),
        include_groups=False
    )
)

print("Total underlying observations:",
      f"{len(configuration_check):,}")

print("Complete unique configurations:",
      f"{configuration_check.sum():,}")

print("Incomplete or duplicate configurations:",
      f"{(~configuration_check).sum():,}")


# ------------------------------------------------
# 5. MEASUREMENT CONSISTENCY
# ------------------------------------------------

print("\n5. MEASUREMENT CONSISTENCY")

measurement_consistency = (
    df_smoke.groupby(key_b)
    .agg(
        PA_unique=("PA", "nunique"),
        ref_unique=("ref", "nunique"),
        RH_unique=("RH", "nunique")
    )
)

for variable in ["PA", "ref", "RH"]:

    inconsistent = (
        measurement_consistency[
            f"{variable}_unique"
        ] > 1
    ).sum()

    print(
        f"{variable} inconsistent observations:",
        f"{inconsistent:,}"
    )


# ------------------------------------------------
# 6. MISSINGNESS
# ------------------------------------------------

print("\n6. MISSINGNESS IN DATASET B")

missing_b = pd.DataFrame({
    "Missing_count": df_smoke.isna().sum(),
    "Missing_percent": (
        df_smoke.isna().mean() * 100
    ).round(3)
}).sort_values("Missing_percent", ascending=False)

display(missing_b)

print("\nMissingness by withholding type:")

missing_by_withholding = (
    df_smoke.groupby("wholdtype")
    .agg(
        Rows=("PA", "size"),

        State_missing_pct=(
            "State",
            lambda x: round(x.isna().mean() * 100, 3)
        ),

        week_missing_pct=(
            "week",
            lambda x: round(x.isna().mean() * 100, 3)
        ),

        PA_missing_pct=(
            "PA",
            lambda x: round(x.isna().mean() * 100, 3)
        ),

        ref_missing_pct=(
            "ref",
            lambda x: round(x.isna().mean() * 100, 3)
        ),

        adj_missing_pct=(
            "adj",
            lambda x: round(x.isna().mean() * 100, 3)
        )
    )
)

display(missing_by_withholding)


# ------------------------------------------------
# 7. NON-POSITIVE MEASUREMENTS
# ------------------------------------------------

print("\n7. NON-POSITIVE MEASUREMENTS")

# One record per underlying hourly observation.
# Measurements are identical across configurations.

base_smoke = (
    df_smoke.drop_duplicates(subset=key_b)
    .copy()
)

print("Underlying observations:",
      f"{len(base_smoke):,}")

for column in ["PA", "ref"]:

    values = pd.to_numeric(
        base_smoke[column],
        errors="coerce"
    )

    print(
        f"{column} non-positive underlying observations:",
        f"{(values <= 0).sum():,}"
    )


# adj is configuration-specific, so count both
# configuration rows and affected underlying observations.

for column in ["adj"]:

    values = pd.to_numeric(
        df_smoke[column],
        errors="coerce"
    )

    bad_mask = values <= 0

    affected_keys = (
        df_smoke.loc[bad_mask, key_b]
        .drop_duplicates()
    )

    print(
        f"\n{column} non-positive configuration rows:",
        f"{bad_mask.sum():,}"
    )

    print(
        f"{column} distinct underlying observations affected:",
        f"{len(affected_keys):,}"
    )


# ------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 75)
print("FINAL VERIFICATION SUMMARY")
print("=" * 75)

print("\nDATASET A")
print("Rows:", f"{len(df_24hr):,}")
print("Distinct sensor-date-region groups:",
      f"{len(a_groups):,}")
print("Repeated groups:",
      f"{(a_groups.size() > 1).sum():,}")
print("Exact duplicates:",
      f"{df_24hr.duplicated().sum():,}")
print("Non-positive reference rows:",
      f"{bad_a.sum():,}")

print("\nDATASET B")
print("Rows:", f"{len(df_smoke):,}")
print("Underlying hourly observations:",
      f"{len(b_groups):,}")
print("Complete 12-configuration observations:",
      f"{configuration_check.sum():,}")
print("Inconsistent PA/ref/RH measurements:",
      f"{sum((measurement_consistency[f'{v}_unique'] > 1).sum() for v in ['PA', 'ref', 'RH']):,}")
print("Exact duplicates:",
      f"{df_smoke.duplicated().sum():,}")

print("\nFINAL CHECK COMPLETE")
print("No original records were modified or removed.")



DATASET A — 24-HOUR COLLOCATION

1. DATASET STRUCTURE
Rows: 12,430
Columns: 31
Exact duplicate rows: 0
Date range: 2017-09-30 to 2020-01-13

2. OBSERVATION STRUCTURE
Distinct ID-Date-region combinations: 9,707
Repeated ID-Date-region combinations: 2,199
Rows belonging to repeated groups: 4,922

3. MISSINGNESS


,Missing_count,Missing_percent
Dewpoint,184,1.480
TempC,184,1.480
ratioPARHcor_FM,166,1.335
ratiocf1_FM,149,1.199
ratiocfatm_FM,149,1.199
diffPARHcor_FM,17,0.137
PM25PARHcor,17,0.137
RH,17,0.137
Unnamed: 0,0,0.000
b0.5to1,0,0.000



4. NON-POSITIVE REFERENCE VALUES
Non-positive PM25FM rows: 44
Distinct ID-Date-region groups affected: 44

5. ENVIRONMENTAL MISSINGNESS


,Missing_count,Missing_percent
RH,17,0.137
TempC,184,1.480
Dewpoint,184,1.480


Rows missing both RH and TempC: 17
Rows missing TempC only: 167
Rows missing RH only: 0

DATASET B — HOURLY WILDFIRE-SMOKE

1. DATASET STRUCTURE
Rows: 1,513,776
Original dataset columns: 13
Current working dataframe columns: 16
Exact duplicate rows: 0
Additional working columns: ['hr_parsed', 'adj_error', 'abs_adj_error']

2. TIMESTAMP VERIFICATION

Timestamp column: hr
Parse failures: 0
Minimum: 2018-01-01 00:00:00
Maximum: 2020-10-20 11:00:00

Timestamp column: hr_parsed
Parse failures: 0
Minimum: 2018-01-01 00:00:00
Maximum: 2020-10-20 11:00:00

Timestamp agreement:
Same_timestamp
True    1513776
Name: count, dtype: int64

Rows with different timestamps: 0

RESULT: hr and hr_parsed agree for all rows.

3. UNDERLYING OBSERVATION STRUCTURE
Distinct city-sensor-time combinations: 126,148

Rows per underlying observation:


,Number_of_observations
Rows_per_observation,
12,126148


Expected rows per observation: 12
Observations with exactly 12 rows: 126,148
Observations with other row counts: 0

4. CONFIGURATION COMPLETENESS
Total underlying observations: 126,148
Complete unique configurations: 126,148
Incomplete or duplicate configurations: 0

5. MEASUREMENT CONSISTENCY
PA inconsistent observations: 0
ref inconsistent observations: 0
RH inconsistent observations: 0

6. MISSINGNESS IN DATASET B


,Missing_count,Missing_percent
State,756888,50.0
week,756888,50.0
Unnamed: 0,0,0.0
hr,0,0.0
city,0,0.0
id,0,0.0
RH,0,0.0
PA,0,0.0
ref,0,0.0
monitortype,0,0.0



Missingness by withholding type:


,Rows,State_missing_pct,week_missing_pct,PA_missing_pct,ref_missing_pct,adj_missing_pct
wholdtype,,,,,,
LOBD,756888,100.0,0.0,0.0,0.0,0.0
LOSO,756888,0.0,100.0,0.0,0.0,0.0



7. NON-POSITIVE MEASUREMENTS
Underlying observations: 126,148
PA non-positive underlying observations: 58
ref non-positive underlying observations: 4,916

adj non-positive configuration rows: 3,790
adj distinct underlying observations affected: 1,881

FINAL VERIFICATION SUMMARY

DATASET A
Rows: 12,430
Distinct sensor-date-region groups: 9,707
Repeated groups: 2,199
Exact duplicates: 0
Non-positive reference rows: 44

DATASET B
Rows: 1,513,776
Underlying hourly observations: 126,148
Complete 12-configuration observations: 126,148
Inconsistent PA/ref/RH measurements: 0
Exact duplicates: 0

FINAL CHECK COMPLETE
No original records were modified or removed.


In [75]:
print("Original hr range:")
print(df_smoke["hr"].min(), "to", df_smoke["hr"].max())

print("\nParsed timestamp range:")
print(df_smoke["hr_parsed"].min(), "to", df_smoke["hr_parsed"].max())

hr_original = pd.to_datetime(df_smoke["hr"], errors="coerce")
hr_parsed = pd.to_datetime(df_smoke["hr_parsed"], errors="coerce")

print("\nTimestamp comparison:")
print("Matching timestamps:", (hr_original == hr_parsed).sum())
print("Mismatching timestamps:", (hr_original != hr_parsed).sum())
print("Missing original timestamps:", hr_original.isna().sum())
print("Missing parsed timestamps:", hr_parsed.isna().sum())

print("\nExamples of mismatches:")
print(
    df_smoke.loc[
        hr_original != hr_parsed,
        ["hr", "hr_parsed"]
    ].head(10).to_string(index=False)
)

Original hr range:
2018-01-01 00:00:00 to 2020-10-20 11:00:00

Parsed timestamp range:
2018-01-01 00:00:00 to 2020-10-20 11:00:00

Timestamp comparison:
Matching timestamps: 1513776
Mismatching timestamps: 0
Missing original timestamps: 0
Missing parsed timestamps: 0

Examples of mismatches:
Empty DataFrame
Columns: [hr, hr_parsed]
Index: []
